In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# TCGA-LIHC
# Strict Development / Holdout Validation
#
# Development = 238
# Holdout Test = 102
#
# IMPORTANT:
# Holdout data are LOCKED during Phase A.
# ============================================================

from pathlib import Path
from datetime import datetime
import json
import shutil
import hashlib
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 路径
# ------------------------------------------------------------

ROOT = Path(_hcc_path(''))

SOURCE_DIR = ROOT / "LIHC_RNA_Step15_Machine_Learning"

PROJECT_DIR = ROOT / "LIHC_RNA_Step15_Machine_Learning_Holdout_Validation"

INPUT_DIR = PROJECT_DIR / "Input"
RESULTS_DIR = PROJECT_DIR / "Results"
FIGURES_DIR = PROJECT_DIR / "Figures"
MODELS_DIR = PROJECT_DIR / "Models"
AUDIT_DIR = PROJECT_DIR / "Audit"
CHECKPOINT_DIR = PROJECT_DIR / "Checkpoint"
NOTEBOOK_DIR = PROJECT_DIR / "Notebook"

for d in [
    INPUT_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    MODELS_DIR,
    AUDIT_DIR,
    CHECKPOINT_DIR,
    NOTEBOOK_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

RUN_TIME = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("=" * 80)
print("TCGA-LIHC Strict Development / Holdout Validation")
print("=" * 80)
print("Source :", SOURCE_DIR)
print("Project:", PROJECT_DIR)
print("Run    :", RUN_TIME)

In [ ]:
# ============================================================
# CONFIG
# ============================================================

RANDOM_STATE = 20260921

# Development repeated CV
N_SPLITS = 5
N_REPEATS = 5

# Inner CV for ElasticNet tuning
INNER_SPLITS = 3

# ElasticNet
L1_RATIO = 0.50

# alpha grid
ALPHA_GRID = np.logspace(
    -4,
    -0.5,
    20
)

# RSF
RUN_RSF = True

RSF_N_ESTIMATORS = 300
RSF_MIN_SAMPLES_SPLIT = 10
RSF_MIN_SAMPLES_LEAF = 5
RSF_MAX_FEATURES = "sqrt"

RSF_PERM_REPEATS = 5
RSF_TOP_N = 10

# Bootstrap：最终 Holdout 时用
N_BOOTSTRAP = 2000

# ------------------------------------------------------------
# ★★★ 测试集保险锁 ★★★
# Phase A 保持 False
# ------------------------------------------------------------

UNLOCK_HOLDOUT = False

VERSION = "Strict_Holdout_v1.0"

CONFIG = {
    "Version": VERSION,
    "Random_State": RANDOM_STATE,
    "Outer_CV": f"{N_REPEATS}x{N_SPLITS}",
    "Inner_CV": INNER_SPLITS,
    "L1_Ratio": L1_RATIO,
    "RSF_Trees": RSF_N_ESTIMATORS,
    "RSF_Top_N": RSF_TOP_N,
    "Bootstrap": N_BOOTSTRAP,
    "Holdout_Unlocked": UNLOCK_HOLDOUT,
    "Run_Time": RUN_TIME,
}

with open(
    AUDIT_DIR / "RUN_CONFIG.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        CONFIG,
        f,
        ensure_ascii=False,
        indent=2
    )

print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

In [ ]:
# ============================================================
# COPY REQUIRED INPUTS
# ============================================================

required_files = {
    "LIHC_Multilevel_Master_Feature_Table.csv":
        SOURCE_DIR / "Input" / "LIHC_Multilevel_Master_Feature_Table.csv",

    "LIHC_Survival_Train_Test_Split_70_30.csv":
        SOURCE_DIR / "Input" / "LIHC_Survival_Train_Test_Split_70_30.csv",

    "Stable_Prognostic_Genes_10.csv":
        SOURCE_DIR / "Results" / "Stable_Prognostic_Genes_10.csv",

    "Stable_Biological_Features.csv":
        SOURCE_DIR / "Results" / "Stable_Biological_Features.csv",

    "Gene_MultiMethod_Selection_Integrated.csv":
        SOURCE_DIR / "Results" / "Gene_MultiMethod_Selection_Integrated.csv",

    "Biological_MultiMethod_Selection_Integrated.csv":
        SOURCE_DIR / "Results" / "Biological_MultiMethod_Selection_Integrated.csv",
}


records = []

for filename, src in required_files.items():

    if not src.exists():
        raise FileNotFoundError(
            f"找不到文件：{src}"
        )

    dst = INPUT_DIR / filename

    if not dst.exists():
        shutil.copy2(src, dst)

    records.append({
        "File": filename,
        "Source": str(src),
        "Destination": str(dst),
        "Size_KB": round(dst.stat().st_size / 1024, 2)
    })

manifest = pd.DataFrame(records)

manifest.to_csv(
    AUDIT_DIR / "INPUT_MANIFEST.csv",
    index=False,
    encoding="utf-8-sig"
)

display(manifest)

In [ ]:
# ============================================================
# LOAD DATA
# ============================================================

MASTER_FILE = INPUT_DIR / "LIHC_Multilevel_Master_Feature_Table.csv"
SPLIT_FILE = INPUT_DIR / "LIHC_Survival_Train_Test_Split_70_30.csv"

master = pd.read_csv(MASTER_FILE)
split_df = pd.read_csv(SPLIT_FILE)

print("Master shape:", master.shape)
print("Split shape :", split_df.shape)

print("\nMaster columns:")
print(master.columns.tolist())

print("\nSplit columns:")
print(split_df.columns.tolist())

In [ ]:
# ============================================================
# IDENTIFY ID / DATASET
# ============================================================

def find_id_column(df):

    candidates = [
        "Sample_ID",
        "sample_id",
        "Sample",
        "ID"
    ]

    for c in candidates:
        if c in df.columns:
            return c

    raise KeyError(
        f"无法识别 Sample ID 列：{list(df.columns)}"
    )


MASTER_ID = find_id_column(master)
SPLIT_ID = find_id_column(split_df)

if "Dataset" not in split_df.columns:
    raise KeyError(
        "历史划分列 Dataset 不存在，停止。"
    )

print("Master ID:", MASTER_ID)
print("Split ID :", SPLIT_ID)

print("\nDataset:")
print(
    split_df["Dataset"]
    .value_counts(dropna=False)
)

In [ ]:
# ============================================================
# BUILD HISTORICAL 340-SAMPLE COHORT
# ============================================================

master[MASTER_ID] = (
    master[MASTER_ID]
    .astype(str)
    .str.strip()
)

split_df[SPLIT_ID] = (
    split_df[SPLIT_ID]
    .astype(str)
    .str.strip()
)

if master[MASTER_ID].duplicated().any():
    raise RuntimeError(
        "Master Sample_ID 存在重复，停止。"
    )

if split_df[SPLIT_ID].duplicated().any():
    raise RuntimeError(
        "Split Sample_ID 存在重复，停止。"
    )


analysis_df = split_df[
    [SPLIT_ID, "Dataset"]
].merge(
    master,
    left_on=SPLIT_ID,
    right_on=MASTER_ID,
    how="left",
    validate="one_to_one"
)

missing_master = analysis_df[
    MASTER_ID
].isna().sum()

print("Matched N:", len(analysis_df))
print("Missing master:", missing_master)

if missing_master != 0:
    raise RuntimeError(
        "存在历史样本无法匹配 Master，停止。"
    )

if len(analysis_df) != 340:
    raise RuntimeError(
        f"历史分析队列应为340，实际={len(analysis_df)}"
    )

print("\nDataset distribution:")
print(
    analysis_df["Dataset"]
    .value_counts()
)

In [ ]:
# ============================================================
# STRICT DEVELOPMENT / HOLDOUT SPLIT
# ============================================================

dataset_norm = (
    analysis_df["Dataset"]
    .astype(str)
    .str.strip()
    .str.lower()
)

development_df = analysis_df[
    dataset_norm == "train"
].copy()

holdout_df = analysis_df[
    dataset_norm == "test"
].copy()


print("=" * 70)
print("STRICT SPLIT")
print("=" * 70)

print("Development Train:", len(development_df))
print("Holdout Test      :", len(holdout_df))


if len(development_df) != 238:
    raise RuntimeError(
        f"Development 应为238，实际={len(development_df)}"
    )

if len(holdout_df) != 102:
    raise RuntimeError(
        f"Holdout 应为102，实际={len(holdout_df)}"
    )

overlap = set(
    development_df[MASTER_ID]
).intersection(
    set(holdout_df[MASTER_ID])
)

if len(overlap) != 0:
    raise RuntimeError(
        "Train/Test 出现样本重叠！"
    )

print("\nPASS")
print("Development = 238")
print("Holdout     = 102")
print("Overlap     = 0")

In [ ]:
# ============================================================
# SURVIVAL OUTCOME AUDIT
# ============================================================

TIME_COL = "OS_time"
EVENT_COL = "OS"

for df_name, df in [
    ("Development", development_df),
    ("Holdout", holdout_df)
]:

    if TIME_COL not in df.columns:
        raise KeyError(f"{df_name}: 缺少 OS_time")

    if EVENT_COL not in df.columns:
        raise KeyError(f"{df_name}: 缺少 OS")

    print(
        df_name,
        "N =", len(df),
        "| Events =",
        pd.to_numeric(
            df[EVENT_COL],
            errors="coerce"
        ).sum(),
        "| Missing OS =",
        df[EVENT_COL].isna().sum(),
        "| Missing time =",
        df[TIME_COL].isna().sum()
    )

In [ ]:
# ============================================================
# CLINICAL NUMERIC ENCODING
# ============================================================

def add_clinical_numeric_columns(df):

    df = df.copy()

    # Age
    if "Age" not in df.columns:
        if "age" in df.columns:
            df["Age"] = pd.to_numeric(
                df["age"],
                errors="coerce"
            )

    # Gender
    if "Gender_binary" not in df.columns:

        if "Gender" in df.columns:

            gender = (
                df["Gender"]
                .astype(str)
                .str.lower()
            )

            df["Gender_binary"] = np.where(
                gender == "male",
                1,
                np.where(
                    gender == "female",
                    0,
                    np.nan
                )
            )

    # Stage
    if "Stage_numeric" not in df.columns and "Stage" in df.columns:

        s = (
            df["Stage"]
            .astype(str)
            .str.upper()
        )

        df["Stage_numeric"] = np.select(
            [
                s.str.contains("IV", na=False),
                s.str.contains("III", na=False),
                s.str.contains("II", na=False),
                s.str.contains("I", na=False),
            ],
            [4, 3, 2, 1],
            default=np.nan
        )

    # T
    if "T_numeric" not in df.columns and "T" in df.columns:

        s = (
            df["T"]
            .astype(str)
            .str.upper()
        )

        df["T_numeric"] = np.select(
            [
                s.str.startswith("T4"),
                s.str.startswith("T3"),
                s.str.startswith("T2"),
                s.str.startswith("T1"),
            ],
            [4, 3, 2, 1],
            default=np.nan
        )

    # N
    if "N_numeric" not in df.columns and "N" in df.columns:

        s = (
            df["N"]
            .astype(str)
            .str.upper()
        )

        df["N_numeric"] = np.select(
            [
                s.str.startswith("N1"),
                s.str.startswith("N0"),
            ],
            [1, 0],
            default=np.nan
        )

    # M
    if "M_numeric" not in df.columns and "M" in df.columns:

        s = (
            df["M"]
            .astype(str)
            .str.upper()
        )

        df["M_numeric"] = np.select(
            [
                s.str.startswith("M1"),
                s.str.startswith("M0"),
            ],
            [1, 0],
            default=np.nan
        )

    return df


development_df = add_clinical_numeric_columns(
    development_df
)

holdout_df = add_clinical_numeric_columns(
    holdout_df
)

CLINICAL_FEATURES = [
    "Age",
    "Gender_binary",
    "Stage_numeric",
    "T_numeric",
    "N_numeric",
    "M_numeric",
]

print(
    development_df[
        CLINICAL_FEATURES
    ].isna().sum()
)

In [ ]:
# ============================================================
# LOCK GENE POOL
# ============================================================

ORIGINAL_TOP10 = [
    "HOXD9",
    "RFESD",
    "RPSAP53",
    "TRGC1",
    "FAM227B",
    "AC015871.6",
    "LINC00324",
    "ACOXL",
    "PTMAP4",
    "ANKRD20A7P",
]

HUB18 = [
    "CENPA",
    "KIF2C",
    "CCNB1",
    "CDC20",
    "PRR11",
    "CDCA8",
    "SLC52A2",
    "RPL8",
    "BOP1",
    "ZFAS1",
    "SNHG6",
    "RPL7P9",
    "CYP8B1",
    "GLYATL1",
    "PLG",
    "ACSM2A",
    "ABAT",
    "MAT1A",
]

GENE_POOL = list(
    dict.fromkeys(
        ORIGINAL_TOP10 + HUB18
    )
)

assert len(GENE_POOL) == 28

missing = [
    x for x in GENE_POOL
    if x not in development_df.columns
]

if missing:
    raise KeyError(
        f"缺失基因：{missing}"
    )

print("Gene pool =", len(GENE_POOL))

In [ ]:
# ============================================================
# LOCK BIOLOGICAL FEATURE POOL
# ============================================================

bio_integrated = pd.read_csv(
    INPUT_DIR / "Biological_MultiMethod_Selection_Integrated.csv"
)

print(bio_integrated.columns.tolist())

feature_col_candidates = [
    "Feature",
    "feature",
    "Variable",
    "variable"
]

BIO_FEATURE_COL = None

for c in feature_col_candidates:
    if c in bio_integrated.columns:
        BIO_FEATURE_COL = c
        break

if BIO_FEATURE_COL is None:
    raise KeyError(
        "无法识别 biological feature 名称列。"
    )

BIO_POOL = (
    bio_integrated[BIO_FEATURE_COL]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

print("Biological pool =", len(BIO_POOL))

if len(BIO_POOL) != 86:
    raise RuntimeError(
        f"Biological pool 应为86，实际={len(BIO_POOL)}"
    )

missing_bio = [
    x for x in BIO_POOL
    if x not in development_df.columns
]

if missing_bio:
    raise KeyError(
        f"Master 缺少 biological features：{missing_bio}"
    )

print("BIO POOL PASS")

In [ ]:
# ============================================================
# LOCK STEP15A FEATURE DEFINITIONS
# ============================================================

sg = pd.read_csv(
    INPUT_DIR / "Stable_Prognostic_Genes_10.csv"
)

sb = pd.read_csv(
    INPUT_DIR / "Stable_Biological_Features.csv"
)

gene_col = (
    "Gene"
    if "Gene" in sg.columns
    else sg.columns[0]
)

bio_col = (
    "Feature"
    if "Feature" in sb.columns
    else sb.columns[0]
)

STABLE_GENES10 = (
    sg[gene_col]
    .dropna()
    .astype(str)
    .tolist()
)

STABLE_BIO7 = (
    sb[bio_col]
    .dropna()
    .astype(str)
    .tolist()
)

GENES10_BIO7 = list(
    dict.fromkeys(
        STABLE_GENES10 + STABLE_BIO7
    )
)

print("StableGenes10:", STABLE_GENES10)
print()
print("StableBio7:", STABLE_BIO7)
print()
print("Genes10_Bio7 N =", len(GENES10_BIO7))

assert len(STABLE_GENES10) == 10
assert len(STABLE_BIO7) == 7
assert len(GENES10_BIO7) == 17

In [ ]:
# ============================================================
# IMPORT MODELING PACKAGES
# ============================================================

import matplotlib.pyplot as plt

from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    StratifiedKFold
)

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance

from lifelines import CoxPHFitter

from sksurv.util import Surv

from sksurv.linear_model import (
    CoxPHSurvivalAnalysis,
    CoxnetSurvivalAnalysis
)

from sksurv.ensemble import (
    RandomSurvivalForest
)

from sksurv.metrics import (
    concordance_index_censored,
    cumulative_dynamic_auc
)

print("Modeling environment PASS")

In [ ]:
# ============================================================
# BASIC FUNCTIONS
# ============================================================

def make_y(df):

    return Surv.from_arrays(
        event=pd.to_numeric(
            df[EVENT_COL],
            errors="coerce"
        ).astype(bool).values,

        time=pd.to_numeric(
            df[TIME_COL],
            errors="coerce"
        ).astype(float).values
    )


def cindex_from_risk(df, risk):

    return concordance_index_censored(
        pd.to_numeric(
            df[EVENT_COL]
        ).astype(bool).values,

        pd.to_numeric(
            df[TIME_COL]
        ).astype(float).values,

        np.asarray(risk)
    )[0]


def preprocess_train_valid(
    train_df,
    valid_df,
    features,
    scale=True
):

    X_train = train_df[
        features
    ].apply(
        pd.to_numeric,
        errors="coerce"
    )

    X_valid = valid_df[
        features
    ].apply(
        pd.to_numeric,
        errors="coerce"
    )

    imputer = SimpleImputer(
        strategy="median"
    )

    X_train = imputer.fit_transform(
        X_train
    )

    X_valid = imputer.transform(
        X_valid
    )

    if scale:

        scaler = StandardScaler()

        X_train = scaler.fit_transform(
            X_train
        )

        X_valid = scaler.transform(
            X_valid
        )

    else:

        scaler = None

    return (
        X_train,
        X_valid,
        imputer,
        scaler
    )

In [ ]:
# ============================================================
# UNIVARIATE COX
# ============================================================

def univariate_cox_selection(
    train_df,
    features,
    p_threshold=0.05
):

    records = []

    for feature in features:

        tmp = pd.DataFrame({
            feature: pd.to_numeric(
                train_df[feature],
                errors="coerce"
            ),
            "OS_time": train_df[TIME_COL],
            "OS": train_df[EVENT_COL],
        }).dropna()

        if (
            len(tmp) < 30
            or tmp[feature].nunique() < 2
        ):
            continue

        try:

            scaler = StandardScaler()

            tmp[feature] = scaler.fit_transform(
                tmp[[feature]]
            )

            cph = CoxPHFitter()

            cph.fit(
                tmp,
                duration_col="OS_time",
                event_col="OS"
            )

            s = cph.summary.loc[
                feature
            ]

            records.append({
                "Feature": feature,
                "Coef": s["coef"],
                "HR": np.exp(s["coef"]),
                "P": s["p"],
            })

        except Exception:

            continue

    result = pd.DataFrame(records)

    if result.empty:
        return result

    result["Selected"] = (
        result["P"] < p_threshold
    )

    return result.sort_values("P")

In [ ]:
# ============================================================
# INNER-CV ELASTIC NET TUNING
# ============================================================

def tune_elastic_alpha(
    train_df,
    features,
    random_state
):

    if len(features) == 0:
        return np.nan

    events = (
        train_df[EVENT_COL]
        .astype(int)
        .values
    )

    cv = StratifiedKFold(
        n_splits=INNER_SPLITS,
        shuffle=True,
        random_state=random_state
    )

    scores = {
        alpha: []
        for alpha in ALPHA_GRID
    }

    for tr_idx, va_idx in cv.split(
        train_df,
        events
    ):

        tr = train_df.iloc[
            tr_idx
        ]

        va = train_df.iloc[
            va_idx
        ]

        X_tr, X_va, _, _ = (
            preprocess_train_valid(
                tr,
                va,
                features,
                scale=True
            )
        )

        y_tr = make_y(tr)

        for alpha in ALPHA_GRID:

            try:

                model = CoxnetSurvivalAnalysis(
                    l1_ratio=L1_RATIO,
                    alphas=[alpha],
                    max_iter=100000
                )

                model.fit(
                    X_tr,
                    y_tr
                )

                risk = model.predict(
                    X_va
                )

                c = cindex_from_risk(
                    va,
                    risk
                )

                scores[alpha].append(c)

            except Exception:
                pass

    summary = []

    for alpha, vals in scores.items():

        if len(vals) > 0:

            summary.append({
                "alpha": alpha,
                "mean_cindex": np.mean(vals)
            })

    if len(summary) == 0:
        return np.nan

    score_df = pd.DataFrame(
        summary
    )

    best = (
        score_df
        .sort_values(
            ["mean_cindex", "alpha"],
            ascending=[
                False,
                False
            ]
        )
        .iloc[0]
    )

    return float(
        best["alpha"]
    )

In [ ]:
# ============================================================
# ELASTIC NET SELECTION
# ============================================================

def elasticnet_selection(
    train_df,
    features,
    random_state
):

    if len(features) == 0:

        return (
            pd.DataFrame(),
            np.nan
        )

    best_alpha = tune_elastic_alpha(
        train_df,
        features,
        random_state
    )

    if np.isnan(best_alpha):

        return (
            pd.DataFrame(),
            np.nan
        )

    X, _, _, _ = preprocess_train_valid(
        train_df,
        train_df,
        features,
        scale=True
    )

    y = make_y(
        train_df
    )

    model = CoxnetSurvivalAnalysis(
        l1_ratio=L1_RATIO,
        alphas=[best_alpha],
        max_iter=100000
    )

    model.fit(
        X,
        y
    )

    coef = np.asarray(
        model.coef_
    ).reshape(-1)

    result = pd.DataFrame({
        "Feature": features,
        "Coefficient": coef
    })

    result[
        "Abs_Coefficient"
    ] = result[
        "Coefficient"
    ].abs()

    result[
        "Selected"
    ] = (
        result[
            "Abs_Coefficient"
        ] > 1e-8
    )

    return (
        result.sort_values(
            "Abs_Coefficient",
            ascending=False
        ),
        best_alpha
    )

In [ ]:
# ============================================================
# RSF PERMUTATION IMPORTANCE
# ============================================================

def rsf_selection(
    train_df,
    features,
    random_state
):

    if len(features) == 0:
        return pd.DataFrame()

    X, _, _, _ = preprocess_train_valid(
        train_df,
        train_df,
        features,
        scale=False
    )

    y = make_y(
        train_df
    )

    model = RandomSurvivalForest(
        n_estimators=RSF_N_ESTIMATORS,
        min_samples_split=RSF_MIN_SAMPLES_SPLIT,
        min_samples_leaf=RSF_MIN_SAMPLES_LEAF,
        max_features=RSF_MAX_FEATURES,
        random_state=random_state,
        n_jobs=-1
    )

    model.fit(
        X,
        y
    )

    perm = permutation_importance(
        model,
        X,
        y,
        n_repeats=RSF_PERM_REPEATS,
        random_state=random_state,
        n_jobs=-1
    )

    result = pd.DataFrame({
        "Feature": features,
        "Importance": perm.importances_mean,
        "Importance_SD": perm.importances_std
    })

    result = (
        result
        .sort_values(
            "Importance",
            ascending=False
        )
        .reset_index(drop=True)
    )

    result["Selected"] = False

    top_n = min(
        RSF_TOP_N,
        len(result)
    )

    if top_n > 0:

        result.loc[
            :top_n-1,
            "Selected"
        ] = True

    return result

In [ ]:
# ============================================================
# RSF PERMUTATION IMPORTANCE
# ============================================================

def rsf_selection(
    train_df,
    features,
    random_state
):

    if len(features) == 0:
        return pd.DataFrame()

    X, _, _, _ = preprocess_train_valid(
        train_df,
        train_df,
        features,
        scale=False
    )

    y = make_y(
        train_df
    )

    model = RandomSurvivalForest(
        n_estimators=RSF_N_ESTIMATORS,
        min_samples_split=RSF_MIN_SAMPLES_SPLIT,
        min_samples_leaf=RSF_MIN_SAMPLES_LEAF,
        max_features=RSF_MAX_FEATURES,
        random_state=random_state,
        n_jobs=-1
    )

    model.fit(
        X,
        y
    )

    perm = permutation_importance(
        model,
        X,
        y,
        n_repeats=RSF_PERM_REPEATS,
        random_state=random_state,
        n_jobs=-1
    )

    result = pd.DataFrame({
        "Feature": features,
        "Importance": perm.importances_mean,
        "Importance_SD": perm.importances_std
    })

    result = (
        result
        .sort_values(
            "Importance",
            ascending=False
        )
        .reset_index(drop=True)
    )

    result["Selected"] = False

    top_n = min(
        RSF_TOP_N,
        len(result)
    )

    if top_n > 0:

        result.loc[
            :top_n-1,
            "Selected"
        ] = True

    return result

In [ ]:
# ============================================================
# MULTI-METHOD TRAIN-ONLY FEATURE SELECTION
# ============================================================

def multimethod_selection(
    train_df,
    feature_pool,
    random_state
):

    # -------------------------------------------
    # 1. Uni-Cox
    # -------------------------------------------

    uni = univariate_cox_selection(
        train_df,
        feature_pool
    )

    if uni.empty:

        uni_selected = []

    else:

        uni_selected = (
            uni.loc[
                uni["Selected"],
                "Feature"
            ]
            .tolist()
        )

    # -------------------------------------------
    # 2. ElasticNet
    # 只在 Uni-Cox candidates 上运行
    # -------------------------------------------

    en, best_alpha = (
        elasticnet_selection(
            train_df,
            uni_selected,
            random_state
        )
    )

    en_selected = (
        en.loc[
            en["Selected"],
            "Feature"
        ].tolist()
        if not en.empty
        else []
    )

    # -------------------------------------------
    # 3. RSF
    # 同样只在 Uni-Cox candidates 上运行
    # -------------------------------------------

    if RUN_RSF:

        rsf = rsf_selection(
            train_df,
            uni_selected,
            random_state
        )

        rsf_selected = (
            rsf.loc[
                rsf["Selected"],
                "Feature"
            ].tolist()
            if not rsf.empty
            else []
        )

    else:

        rsf = pd.DataFrame()
        rsf_selected = []

    # -------------------------------------------
    # consensus
    # -------------------------------------------

    records = []

    for feature in feature_pool:

        u = int(
            feature in uni_selected
        )

        e = int(
            feature in en_selected
        )

        r = int(
            feature in rsf_selected
        )

        count = u + e + r

        records.append({
            "Feature": feature,
            "UniCox": u,
            "ElasticNet": e,
            "RSF": r,
            "Selection_Count": count,
            "Strict_3of3": int(
                count == 3
            ),
            "Majority_2of3": int(
                count >= 2
            )
        })

    consensus = pd.DataFrame(
        records
    )

    strict = (
        consensus.loc[
            consensus["Strict_3of3"] == 1,
            "Feature"
        ]
        .tolist()
    )

    majority = (
        consensus.loc[
            consensus["Majority_2of3"] == 1,
            "Feature"
        ]
        .tolist()
    )

    return {
        "uni": uni,
        "elasticnet": en,
        "rsf": rsf,
        "consensus": consensus,
        "strict": strict,
        "majority": majority,
        "best_alpha": best_alpha
    }

In [ ]:
# ============================================================
# MODEL FIT / PREDICT FUNCTIONS
# ============================================================

def predict_coxph(
    train_df,
    valid_df,
    features
):

    X_train, X_valid, _, _ = (
        preprocess_train_valid(
            train_df,
            valid_df,
            features,
            scale=True
        )
    )

    model = CoxPHSurvivalAnalysis(
        alpha=0.01
    )

    model.fit(
        X_train,
        make_y(train_df)
    )

    return model.predict(
        X_valid
    )


def predict_elasticnet(
    train_df,
    valid_df,
    features,
    random_state
):

    best_alpha = tune_elastic_alpha(
        train_df,
        features,
        random_state
    )

    X_train, X_valid, _, _ = (
        preprocess_train_valid(
            train_df,
            valid_df,
            features,
            scale=True
        )
    )

    model = CoxnetSurvivalAnalysis(
        l1_ratio=L1_RATIO,
        alphas=[best_alpha],
        max_iter=100000
    )

    model.fit(
        X_train,
        make_y(train_df)
    )

    risk = model.predict(
        X_valid
    )

    return risk


def predict_rsf(
    train_df,
    valid_df,
    features,
    random_state
):

    X_train, X_valid, _, _ = (
        preprocess_train_valid(
            train_df,
            valid_df,
            features,
            scale=False
        )
    )

    model = RandomSurvivalForest(
        n_estimators=RSF_N_ESTIMATORS,
        min_samples_split=RSF_MIN_SAMPLES_SPLIT,
        min_samples_leaf=RSF_MIN_SAMPLES_LEAF,
        max_features=RSF_MAX_FEATURES,
        random_state=random_state,
        n_jobs=-1
    )

    model.fit(
        X_train,
        make_y(train_df)
    )

    return model.predict(
        X_valid
    )

In [ ]:
# ============================================================
# DEVELOPMENT-ONLY REPEATED CV
# ============================================================

outer_cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE
)

development_events = (
    development_df[
        EVENT_COL
    ]
    .astype(int)
    .values
)

CV_SPLITS = []

for i, (
    train_idx,
    valid_idx
) in enumerate(
    outer_cv.split(
        development_df,
        development_events
    )
):

    repeat = (
        i // N_SPLITS
    ) + 1

    fold = (
        i % N_SPLITS
    ) + 1

    CV_SPLITS.append({
        "Repeat": repeat,
        "Fold": fold,
        "Train_idx": train_idx,
        "Valid_idx": valid_idx
    })


print("Development N:", len(development_df))
print(
    "Repeated CV folds:",
    len(CV_SPLITS)
)

first = CV_SPLITS[0]

print(
    "First fold train:",
    len(first["Train_idx"])
)

print(
    "First fold valid:",
    len(first["Valid_idx"])
)

print(
    "Holdout remains locked:",
    len(holdout_df)
)

In [ ]:
# ============================================================
# SAVE DEVELOPMENT CV MANIFEST
# ============================================================

manifest_rows = []

for s in CV_SPLITS:

    r = s["Repeat"]
    f = s["Fold"]

    tr = development_df.iloc[
        s["Train_idx"]
    ]

    va = development_df.iloc[
        s["Valid_idx"]
    ]

    for sid in tr[MASTER_ID]:

        manifest_rows.append({
            "Repeat": r,
            "Fold": f,
            "Sample_ID": sid,
            "Role": "Development_Train"
        })

    for sid in va[MASTER_ID]:

        manifest_rows.append({
            "Repeat": r,
            "Fold": f,
            "Sample_ID": sid,
            "Role": "Development_Validation"
        })


cv_manifest = pd.DataFrame(
    manifest_rows
)

cv_manifest.to_csv(
    RESULTS_DIR /
    "Development_RepeatedCV_Split_Manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    cv_manifest[
        "Role"
    ].value_counts()
)

In [ ]:
# ============================================================
# DEVELOPMENT TRAIN-ONLY FEATURE SELECTION
# WITH CHECKPOINT
# ============================================================

for s in CV_SPLITS:

    repeat = s["Repeat"]
    fold = s["Fold"]

    gene_file = (
        CHECKPOINT_DIR /
        f"R{repeat:02d}_F{fold:02d}_GeneSelection.csv"
    )

    bio_file = (
        CHECKPOINT_DIR /
        f"R{repeat:02d}_F{fold:02d}_BioSelection.csv"
    )

    info_file = (
        CHECKPOINT_DIR /
        f"R{repeat:02d}_F{fold:02d}_Info.json"
    )

    if (
        gene_file.exists()
        and bio_file.exists()
        and info_file.exists()
    ):

        print(
            f"SKIP R{repeat} F{fold}"
        )

        continue

    train_df = development_df.iloc[
        s["Train_idx"]
    ].copy()

    print(
        f"RUN R{repeat} F{fold}"
        f" | Train={len(train_df)}"
    )

    seed = (
        RANDOM_STATE
        + repeat * 100
        + fold
    )

    gene_result = multimethod_selection(
        train_df,
        GENE_POOL,
        seed
    )

    bio_result = multimethod_selection(
        train_df,
        BIO_POOL,
        seed + 10000
    )

    gene_result[
        "consensus"
    ].to_csv(
        gene_file,
        index=False,
        encoding="utf-8-sig"
    )

    bio_result[
        "consensus"
    ].to_csv(
        bio_file,
        index=False,
        encoding="utf-8-sig"
    )

    info = {
        "Repeat": repeat,
        "Fold": fold,
        "N_Train": len(train_df),
        "Gene_Strict_N": len(
            gene_result["strict"]
        ),
        "Bio_Strict_N": len(
            bio_result["strict"]
        ),
        "Gene_Strict": gene_result[
            "strict"
        ],
        "Bio_Strict": bio_result[
            "strict"
        ],
        "Gene_Elastic_Alpha":
            gene_result["best_alpha"],
        "Bio_Elastic_Alpha":
            bio_result["best_alpha"]
    }

    with open(
        info_file,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            info,
            f,
            ensure_ascii=False,
            indent=2
        )

    print(
        f"  Gene strict={len(gene_result['strict'])}"
        f" | Bio strict={len(bio_result['strict'])}"
    )

In [ ]:
# ============================================================
# AGGREGATE SELECTION FREQUENCY
# ============================================================

gene_tables = []
bio_tables = []
fold_info = []

for s in CV_SPLITS:

    r = s["Repeat"]
    f = s["Fold"]

    gene_file = (
        CHECKPOINT_DIR /
        f"R{r:02d}_F{f:02d}_GeneSelection.csv"
    )

    bio_file = (
        CHECKPOINT_DIR /
        f"R{r:02d}_F{f:02d}_BioSelection.csv"
    )

    info_file = (
        CHECKPOINT_DIR /
        f"R{r:02d}_F{f:02d}_Info.json"
    )

    if not (
        gene_file.exists()
        and bio_file.exists()
        and info_file.exists()
    ):
        raise RuntimeError(
            f"缺少 checkpoint: R{r} F{f}"
        )

    g = pd.read_csv(
        gene_file
    )

    b = pd.read_csv(
        bio_file
    )

    g["Repeat"] = r
    g["Fold"] = f

    b["Repeat"] = r
    b["Fold"] = f

    gene_tables.append(g)
    bio_tables.append(b)

    with open(
        info_file,
        "r",
        encoding="utf-8"
    ) as fh:

        fold_info.append(
            json.load(fh)
        )


gene_all = pd.concat(
    gene_tables,
    ignore_index=True
)

bio_all = pd.concat(
    bio_tables,
    ignore_index=True
)


def summarize_frequency(df):

    n_folds = (
        df[
            ["Repeat", "Fold"]
        ]
        .drop_duplicates()
        .shape[0]
    )

    out = (
        df
        .groupby("Feature")
        .agg(
            UniCox_Frequency=(
                "UniCox",
                "mean"
            ),
            ElasticNet_Frequency=(
                "ElasticNet",
                "mean"
            ),
            RSF_Frequency=(
                "RSF",
                "mean"
            ),
            Strict_3of3_Frequency=(
                "Strict_3of3",
                "mean"
            ),
            Majority_2of3_Frequency=(
                "Majority_2of3",
                "mean"
            )
        )
        .reset_index()
    )

    out["N_Folds"] = n_folds

    return out.sort_values(
        "Strict_3of3_Frequency",
        ascending=False
    )


gene_freq = summarize_frequency(
    gene_all
)

bio_freq = summarize_frequency(
    bio_all
)

gene_freq[
    "Feature_Family"
] = "Gene"

bio_freq[
    "Feature_Family"
] = "Biological"

selection_frequency = pd.concat(
    [
        gene_freq,
        bio_freq
    ],
    ignore_index=True
)

selection_frequency.to_csv(
    RESULTS_DIR /
    "Development_Selection_Frequency.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame(
    fold_info
).to_csv(
    RESULTS_DIR /
    "Development_Selection_Summary_ByFold.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    gene_freq.head(20)
)

display(
    bio_freq.head(20)
)

In [ ]:
# ============================================================
# DYNAMIC MODEL PERFORMANCE
# DEVELOPMENT ONLY
# ============================================================

dynamic_records = []

for s in CV_SPLITS:

    r = s["Repeat"]
    f = s["Fold"]

    output_file = (
        CHECKPOINT_DIR /
        f"R{r:02d}_F{f:02d}_DynamicPerformance.csv"
    )

    if output_file.exists():

        print(
            f"SKIP dynamic R{r} F{f}"
        )

        continue

    train_df = development_df.iloc[
        s["Train_idx"]
    ].copy()

    valid_df = development_df.iloc[
        s["Valid_idx"]
    ].copy()

    gene_sel = pd.read_csv(
        CHECKPOINT_DIR /
        f"R{r:02d}_F{f:02d}_GeneSelection.csv"
    )

    bio_sel = pd.read_csv(
        CHECKPOINT_DIR /
        f"R{r:02d}_F{f:02d}_BioSelection.csv"
    )

    gene_features = (
        gene_sel.loc[
            gene_sel["Strict_3of3"] == 1,
            "Feature"
        ].tolist()
    )

    bio_features = (
        bio_sel.loc[
            bio_sel["Strict_3of3"] == 1,
            "Feature"
        ].tolist()
    )

    model_sets = {
        "Dynamic_Gene":
            gene_features,

        "Dynamic_Bio":
            bio_features,

        "Dynamic_Integrated":
            list(
                dict.fromkeys(
                    gene_features
                    + bio_features
                )
            )
    }

    fold_records = []

    for model_name, features in model_sets.items():

        if len(features) == 0:
            continue

        seed = (
            RANDOM_STATE
            + r * 100
            + f
        )

        # CoxPH
        try:

            risk = predict_coxph(
                train_df,
                valid_df,
                features
            )

            fold_records.append({
                "Repeat": r,
                "Fold": f,
                "Model": model_name,
                "Algorithm": "CoxPH",
                "N_Features": len(features),
                "C_Index": cindex_from_risk(
                    valid_df,
                    risk
                )
            })

        except Exception as e:

            print(
                "CoxPH failed:",
                model_name,
                repr(e)
            )

        # ElasticNet
        try:

            risk = predict_elasticnet(
                train_df,
                valid_df,
                features,
                seed
            )

            fold_records.append({
                "Repeat": r,
                "Fold": f,
                "Model": model_name,
                "Algorithm": "ElasticNet",
                "N_Features": len(features),
                "C_Index": cindex_from_risk(
                    valid_df,
                    risk
                )
            })

        except Exception as e:

            print(
                "ElasticNet failed:",
                model_name,
                repr(e)
            )

        # RSF
        if RUN_RSF:

            try:

                risk = predict_rsf(
                    train_df,
                    valid_df,
                    features,
                    seed
                )

                fold_records.append({
                    "Repeat": r,
                    "Fold": f,
                    "Model": model_name,
                    "Algorithm": "RSF",
                    "N_Features": len(features),
                    "C_Index": cindex_from_risk(
                        valid_df,
                        risk
                    )
                })

            except Exception as e:

                print(
                    "RSF failed:",
                    model_name,
                    repr(e)
                )

    pd.DataFrame(
        fold_records
    ).to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"DONE dynamic R{r} F{f}"
    )

In [ ]:
# ============================================================
# AGGREGATE DYNAMIC MODEL PERFORMANCE
# ============================================================

tables = []

for s in CV_SPLITS:

    r = s["Repeat"]
    f = s["Fold"]

    p = (
        CHECKPOINT_DIR /
        f"R{r:02d}_F{f:02d}_DynamicPerformance.csv"
    )

    if p.exists():
        tables.append(
            pd.read_csv(p)
        )

dynamic_perf = pd.concat(
    tables,
    ignore_index=True
)

dynamic_summary = (
    dynamic_perf
    .groupby(
        ["Model", "Algorithm"]
    )
    .agg(
        Mean_C_Index=(
            "C_Index",
            "mean"
        ),
        SD_C_Index=(
            "C_Index",
            "std"
        ),
        Median_C_Index=(
            "C_Index",
            "median"
        ),
        Min_C_Index=(
            "C_Index",
            "min"
        ),
        Max_C_Index=(
            "C_Index",
            "max"
        ),
        Mean_N_Features=(
            "N_Features",
            "mean"
        ),
        N_Folds=(
            "C_Index",
            "count"
        )
    )
    .reset_index()
)

dynamic_perf.to_csv(
    RESULTS_DIR /
    "Development_Dynamic_Performance_ByFold.csv",
    index=False,
    encoding="utf-8-sig"
)

dynamic_summary.to_csv(
    RESULTS_DIR /
    "Development_Dynamic_Model_Summary.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    dynamic_summary.sort_values(
        "Mean_C_Index",
        ascending=False
    )
)

In [ ]:
# ============================================================
# LOCKED FEATURE-SET DEVELOPMENT CV
# ============================================================

FIXED_FEATURE_SETS = {

    "Original_Top10":
        ORIGINAL_TOP10,

    "StableGenes10":
        STABLE_GENES10,

    "StableBio7":
        STABLE_BIO7,

    "Genes10_Bio7":
        GENES10_BIO7,

    "Clinical":
        CLINICAL_FEATURES,

    "Clinical_StableGenes10":
        CLINICAL_FEATURES
        + STABLE_GENES10,

    "Clinical_Genes10_Bio7":
        CLINICAL_FEATURES
        + GENES10_BIO7,
}


for s in CV_SPLITS:

    r = s["Repeat"]
    f = s["Fold"]

    output_file = (
        CHECKPOINT_DIR /
        f"R{r:02d}_F{f:02d}_FixedPerformance.csv"
    )

    if output_file.exists():

        print(
            f"SKIP fixed R{r} F{f}"
        )
        continue

    train_df = development_df.iloc[
        s["Train_idx"]
    ].copy()

    valid_df = development_df.iloc[
        s["Valid_idx"]
    ].copy()

    records = []

    for model_name, features in (
        FIXED_FEATURE_SETS.items()
    ):

        seed = (
            RANDOM_STATE
            + r * 100
            + f
        )

        # CoxPH
        try:

            risk = predict_coxph(
                train_df,
                valid_df,
                features
            )

            records.append({
                "Repeat": r,
                "Fold": f,
                "Model": model_name,
                "Algorithm": "CoxPH",
                "N_Features": len(features),
                "C_Index": cindex_from_risk(
                    valid_df,
                    risk
                )
            })

        except Exception as e:

            print(
                "CoxPH failed:",
                model_name,
                repr(e)
            )

        # ElasticNet
        try:

            risk = predict_elasticnet(
                train_df,
                valid_df,
                features,
                seed
            )

            records.append({
                "Repeat": r,
                "Fold": f,
                "Model": model_name,
                "Algorithm": "ElasticNet",
                "N_Features": len(features),
                "C_Index": cindex_from_risk(
                    valid_df,
                    risk
                )
            })

        except Exception as e:

            print(
                "ElasticNet failed:",
                model_name,
                repr(e)
            )

    pd.DataFrame(
        records
    ).to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"DONE fixed R{r} F{f}"
    )

In [ ]:
# ============================================================
# FIXED MODEL DEVELOPMENT SUMMARY
# ============================================================

tables = []

for s in CV_SPLITS:

    r = s["Repeat"]
    f = s["Fold"]

    p = (
        CHECKPOINT_DIR /
        f"R{r:02d}_F{f:02d}_FixedPerformance.csv"
    )

    if p.exists():

        tables.append(
            pd.read_csv(p)
        )

fixed_perf = pd.concat(
    tables,
    ignore_index=True
)

fixed_summary = (
    fixed_perf
    .groupby(
        ["Model", "Algorithm"]
    )
    .agg(
        Mean_C_Index=(
            "C_Index",
            "mean"
        ),
        SD_C_Index=(
            "C_Index",
            "std"
        ),
        Median_C_Index=(
            "C_Index",
            "median"
        ),
        Min_C_Index=(
            "C_Index",
            "min"
        ),
        Max_C_Index=(
            "C_Index",
            "max"
        ),
        N_Folds=(
            "C_Index",
            "count"
        )
    )
    .reset_index()
)

fixed_perf.to_csv(
    RESULTS_DIR /
    "Development_Fixed_Performance_ByFold.csv",
    index=False,
    encoding="utf-8-sig"
)

fixed_summary.to_csv(
    RESULTS_DIR /
    "Development_Fixed_Model_Summary.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    fixed_summary.sort_values(
        "Mean_C_Index",
        ascending=False
    )
)

In [ ]:
# ============================================================
# FULL DEVELOPMENT FEATURE SELECTION
#
# This defines the candidate dynamic signature
# BEFORE holdout is opened.
# ============================================================

print(
    "Running final feature selection on "
    "FULL DEVELOPMENT N=238..."
)

final_gene_sel = multimethod_selection(
    development_df,
    GENE_POOL,
    RANDOM_STATE + 50000
)

final_bio_sel = multimethod_selection(
    development_df,
    BIO_POOL,
    RANDOM_STATE + 60000
)

FINAL_DYNAMIC_GENES = (
    final_gene_sel[
        "strict"
    ]
)

FINAL_DYNAMIC_BIO = (
    final_bio_sel[
        "strict"
    ]
)

FINAL_DYNAMIC_INTEGRATED = list(
    dict.fromkeys(
        FINAL_DYNAMIC_GENES
        + FINAL_DYNAMIC_BIO
    )
)

print(
    "Dynamic Genes:",
    FINAL_DYNAMIC_GENES
)

print(
    "\nDynamic Bio:",
    FINAL_DYNAMIC_BIO
)

print(
    "\nDynamic Integrated N:",
    len(
        FINAL_DYNAMIC_INTEGRATED
    )
)

final_features = []

for f in FINAL_DYNAMIC_GENES:

    final_features.append({
        "Feature": f,
        "Feature_Family": "Gene",
        "Selected_In_Full_Development": 1
    })

for f in FINAL_DYNAMIC_BIO:

    final_features.append({
        "Feature": f,
        "Feature_Family": "Biological",
        "Selected_In_Full_Development": 1
    })


pd.DataFrame(
    final_features
).to_csv(
    RESULTS_DIR /
    "Development_FullTrain_Dynamic_Features.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# DEVELOPMENT-ONLY FINAL SUMMARY
# ============================================================

development_summary = pd.concat(
    [
        fixed_summary.assign(
            Analysis_Type="Fixed"
        ),
        dynamic_summary.assign(
            Analysis_Type="Dynamic"
        )
    ],
    ignore_index=True
)

development_summary = (
    development_summary
    .sort_values(
        "Mean_C_Index",
        ascending=False
    )
)

development_summary.to_csv(
    RESULTS_DIR /
    "DEVELOPMENT_FINAL_MODEL_COMPARISON.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    development_summary
)

In [ ]:
# ============================================================
# DEVELOPMENT PERFORMANCE FIGURE
# ============================================================

plot_df = pd.concat(
    [
        fixed_perf.assign(
            Analysis_Type="Fixed"
        ),
        dynamic_perf.assign(
            Analysis_Type="Dynamic"
        )
    ],
    ignore_index=True
)

plot_df[
    "Label"
] = (
    plot_df["Model"]
    + " | "
    + plot_df["Algorithm"]
)

order = (
    plot_df
    .groupby("Label")[
        "C_Index"
    ]
    .mean()
    .sort_values()
    .index
)

data = [
    plot_df.loc[
        plot_df["Label"] == label,
        "C_Index"
    ].values
    for label in order
]

fig, ax = plt.subplots(
    figsize=(12, 9)
)

ax.boxplot(
    data,
    vert=False,
    labels=order
)

ax.set_xlabel(
    "Development repeated-CV C-index"
)

ax.set_title(
    "Development Cohort Model Stability (N=238)"
)

fig.tight_layout()

fig.savefig(
    FIGURES_DIR /
    "Development_RepeatedCV_Model_Performance.png",
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    FIGURES_DIR /
    "Development_RepeatedCV_Model_Performance.svg",
    bbox_inches="tight"
)

plt.show()

plt.close(fig)

In [ ]:
# ============================================================
# PHASE A AUDIT
# ============================================================

audit = []

audit.append(
    "TCGA-LIHC Strict Holdout Validation"
)

audit.append(
    "=" * 80
)

audit.append(
    f"Version: {VERSION}"
)

audit.append(
    f"Development N: {len(development_df)}"
)

audit.append(
    f"Development events: "
    f"{development_df[EVENT_COL].sum()}"
)

audit.append(
    f"Holdout N: {len(holdout_df)}"
)

audit.append(
    "Holdout used in Phase A: NO"
)

audit.append(
    ""
)

audit.append(
    f"Repeated CV: {N_REPEATS}x{N_SPLITS}"
)

audit.append(
    f"Gene candidate pool: {len(GENE_POOL)}"
)

audit.append(
    f"Biological candidate pool: {len(BIO_POOL)}"
)

audit.append(
    ""
)

audit.append(
    "Final dynamic genes:"
)

audit.append(
    "; ".join(
        FINAL_DYNAMIC_GENES
    )
)

audit.append(
    ""
)

audit.append(
    "Final dynamic biological features:"
)

audit.append(
    "; ".join(
        FINAL_DYNAMIC_BIO
    )
)

audit.append(
    ""
)

audit.append(
    "PHASE A STATUS: COMPLETE"
)

audit.append(
    "HOLDOUT STATUS: LOCKED"
)

audit_text = "\n".join(
    audit
)

with open(
    AUDIT_DIR /
    "PHASE_A_DEVELOPMENT_AUDIT.md",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        audit_text
    )

print(
    audit_text
)

In [ ]:
# ============================================================
# CELL 32A
# CANDIDATE MODEL REGISTRY
# ============================================================

MODEL_REGISTRY = pd.DataFrame([
    {
        "Model_ID": "M01_OriginalTop10_UnpenalizedCox",
        "Feature_Set": "Original_Top10",
        "N_Features": len(ORIGINAL_TOP10),
        "Algorithm": "CoxPH",
        "Penalty": "None",
        "Role": "Core candidate",
        "Development_Status": "To be audited",
        "Holdout_Status": "LOCKED"
    },
    {
        "Model_ID": "M02_OriginalTop10_RidgeCox",
        "Feature_Set": "Original_Top10",
        "N_Features": len(ORIGINAL_TOP10),
        "Algorithm": "CoxPH",
        "Penalty": "L2 alpha=0.01",
        "Role": "Previously labelled CoxPH",
        "Development_Status": "Existing CV result",
        "Holdout_Status": "LOCKED"
    },
    {
        "Model_ID": "M03_OriginalTop10_ElasticNet",
        "Feature_Set": "Original_Top10",
        "N_Features": len(ORIGINAL_TOP10),
        "Algorithm": "ElasticNet Cox",
        "Penalty": f"L1_ratio={L1_RATIO}",
        "Role": "Regularized benchmark",
        "Development_Status": "Existing CV result",
        "Holdout_Status": "LOCKED"
    },
    {
        "Model_ID": "M04_Dynamic9",
        "Feature_Set": "Development-derived dynamic genes",
        "N_Features": len(FINAL_DYNAMIC_GENES),
        "Algorithm": "Cox/ElasticNet",
        "Penalty": "Pending algorithm lock",
        "Role": "New development candidate",
        "Development_Status": "Selected from Development only",
        "Holdout_Status": "LOCKED"
    },
    {
        "Model_ID": "M05_StableGenes10_ElasticNet",
        "Feature_Set": "StableGenes10",
        "N_Features": len(STABLE_GENES10),
        "Algorithm": "ElasticNet Cox",
        "Penalty": f"L1_ratio={L1_RATIO}",
        "Role": "Step15A stability model",
        "Development_Status": "Existing",
        "Holdout_Status": "LOCKED"
    },
    {
        "Model_ID": "M06_Genes10Bio7_ElasticNet",
        "Feature_Set": "Genes10_Bio7",
        "N_Features": len(GENES10_BIO7),
        "Algorithm": "ElasticNet Cox",
        "Penalty": f"L1_ratio={L1_RATIO}",
        "Role": "Incremental biological model",
        "Development_Status": "Existing",
        "Holdout_Status": "LOCKED"
    }
])

MODEL_REGISTRY.to_csv(
    RESULTS_DIR / "00_Candidate_Model_Registry.csv",
    index=False,
    encoding="utf-8-sig"
)

display(MODEL_REGISTRY)

In [ ]:
# ============================================================
# CELL 32B
# ORIGINAL TOP10
# UNPENALIZED COX vs RIDGE COX vs ELASTICNET
# DEVELOPMENT-ONLY 5x5 CV
# ============================================================

def predict_cox_alpha(
    train_df,
    valid_df,
    features,
    alpha
):
    X_train, X_valid, _, _ = preprocess_train_valid(
        train_df,
        valid_df,
        features,
        scale=True
    )

    model = CoxPHSurvivalAnalysis(
        alpha=alpha
    )

    model.fit(
        X_train,
        make_y(train_df)
    )

    return model.predict(X_valid)


comparison_records = []

for s in CV_SPLITS:

    r = s["Repeat"]
    f = s["Fold"]

    train_df = development_df.iloc[
        s["Train_idx"]
    ].copy()

    valid_df = development_df.iloc[
        s["Valid_idx"]
    ].copy()

    seed = RANDOM_STATE + r * 100 + f

    # ------------------------------
    # True unpenalized Cox
    # ------------------------------
    try:
        risk = predict_cox_alpha(
            train_df,
            valid_df,
            ORIGINAL_TOP10,
            alpha=0.0
        )

        comparison_records.append({
            "Repeat": r,
            "Fold": f,
            "Model": "OriginalTop10_UnpenalizedCox",
            "C_Index": cindex_from_risk(
                valid_df,
                risk
            )
        })

    except Exception as e:
        print(
            f"Unpenalized Cox failed R{r}F{f}:",
            repr(e)
        )

    # ------------------------------
    # Ridge Cox alpha=0.01
    # ------------------------------
    try:
        risk = predict_cox_alpha(
            train_df,
            valid_df,
            ORIGINAL_TOP10,
            alpha=0.01
        )

        comparison_records.append({
            "Repeat": r,
            "Fold": f,
            "Model": "OriginalTop10_RidgeCox_a0.01",
            "C_Index": cindex_from_risk(
                valid_df,
                risk
            )
        })

    except Exception as e:
        print(
            f"Ridge Cox failed R{r}F{f}:",
            repr(e)
        )

    # ------------------------------
    # ElasticNet
    # ------------------------------
    try:
        risk = predict_elasticnet(
            train_df,
            valid_df,
            ORIGINAL_TOP10,
            seed
        )

        comparison_records.append({
            "Repeat": r,
            "Fold": f,
            "Model": "OriginalTop10_ElasticNet",
            "C_Index": cindex_from_risk(
                valid_df,
                risk
            )
        })

    except Exception as e:
        print(
            f"ElasticNet failed R{r}F{f}:",
            repr(e)
        )


original_top10_algorithm_cv = pd.DataFrame(
    comparison_records
)

original_top10_algorithm_summary = (
    original_top10_algorithm_cv
    .groupby("Model")
    ["C_Index"]
    .agg(
        Mean_C_Index="mean",
        SD_C_Index="std",
        Median_C_Index="median",
        Min_C_Index="min",
        Max_C_Index="max",
        N_Folds="count"
    )
    .reset_index()
)

original_top10_algorithm_cv.to_csv(
    RESULTS_DIR /
    "01_OriginalTop10_AlgorithmComparison_ByFold.csv",
    index=False,
    encoding="utf-8-sig"
)

original_top10_algorithm_summary.to_csv(
    RESULTS_DIR /
    "02_OriginalTop10_AlgorithmComparison_Summary.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    original_top10_algorithm_summary
    .sort_values(
        "Mean_C_Index",
        ascending=False
    )
)

In [ ]:
# ============================================================
# CELL 32C
# ALGORITHM COMPARISON FIGURE
# ============================================================

labels = (
    original_top10_algorithm_summary
    .sort_values("Mean_C_Index")
    ["Model"]
    .tolist()
)

plot_data = [
    original_top10_algorithm_cv.loc[
        original_top10_algorithm_cv["Model"] == label,
        "C_Index"
    ].values
    for label in labels
]

fig, ax = plt.subplots(
    figsize=(9, 5.5)
)

ax.boxplot(
    plot_data,
    vert=False,
    labels=labels
)

ax.set_xlabel(
    "Development repeated-CV C-index"
)

ax.set_title(
    "Original Top10: Cox Model Comparison"
)

fig.tight_layout()

fig.savefig(
    FIGURES_DIR /
    "01_OriginalTop10_CoxModel_Comparison.png",
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    FIGURES_DIR /
    "01_OriginalTop10_CoxModel_Comparison.svg",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# CELL 32D
# FIT ORIGINAL TOP10 UNPENALIZED COX
# FULL DEVELOPMENT N=238
# ============================================================

import joblib

# ---------------------------------------------
# Development preprocessing
# ---------------------------------------------

X_dev_raw = (
    development_df[
        ORIGINAL_TOP10
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)

dev_imputer = SimpleImputer(
    strategy="median"
)

X_dev_imp = dev_imputer.fit_transform(
    X_dev_raw
)

dev_scaler = StandardScaler()

X_dev_std = dev_scaler.fit_transform(
    X_dev_imp
)

y_dev = make_y(
    development_df
)

# ---------------------------------------------
# Prediction model
# ---------------------------------------------

original_cox_model = CoxPHSurvivalAnalysis(
    alpha=0.0
)

original_cox_model.fit(
    X_dev_std,
    y_dev
)

dev_risk = original_cox_model.predict(
    X_dev_std
)

dev_cindex = cindex_from_risk(
    development_df,
    dev_risk
)

print(
    "Full Development apparent C-index:",
    round(dev_cindex, 4)
)

In [ ]:
# ============================================================
# CELL 32E
# COEFFICIENT / HR / CI / P-VALUE
# ============================================================

lifelines_df = pd.DataFrame(
    X_dev_std,
    columns=ORIGINAL_TOP10,
    index=development_df.index
)

lifelines_df["OS_time"] = (
    development_df[TIME_COL].values
)

lifelines_df["OS"] = (
    development_df[EVENT_COL].values
)

cph_inference = CoxPHFitter(
    penalizer=0.0
)

cph_inference.fit(
    lifelines_df,
    duration_col="OS_time",
    event_col="OS",
    show_progress=False
)

coef_table = (
    cph_inference.summary
    .reset_index()
)

coef_table = coef_table.rename(
    columns={
        "covariate": "Gene",
        "coef": "Coefficient",
        "exp(coef)": "HR_per_1SD",
        "p": "P_value",
        "coef lower 95%": "Coef_CI_Lower",
        "coef upper 95%": "Coef_CI_Upper",
        "exp(coef) lower 95%": "HR_CI_Lower",
        "exp(coef) upper 95%": "HR_CI_Upper"
    }
)

keep_cols = [
    "Gene",
    "Coefficient",
    "HR_per_1SD",
    "HR_CI_Lower",
    "HR_CI_Upper",
    "P_value"
]

coef_table = coef_table[
    keep_cols
]

coef_table.to_csv(
    RESULTS_DIR /
    "03_OriginalTop10_UnpenalizedCox_Coefficients.csv",
    index=False,
    encoding="utf-8-sig"
)

display(coef_table)

In [ ]:
# ============================================================
# CELL 32F
# PROPORTIONAL HAZARDS ASSUMPTION
# ============================================================

from lifelines.statistics import proportional_hazard_test

ph_test = proportional_hazard_test(
    cph_inference,
    lifelines_df,
    time_transform="rank"
)

ph_table = (
    ph_test.summary
    .reset_index()
)

ph_table = ph_table.rename(
    columns={
        "index": "Gene"
    }
)

ph_table["PH_Violation_P_lt_0.05"] = (
    ph_table["p"] < 0.05
)

ph_table.to_csv(
    RESULTS_DIR /
    "04_OriginalTop10_PH_Assumption_Test.csv",
    index=False,
    encoding="utf-8-sig"
)

display(ph_table)

In [ ]:
# ============================================================
# CELL 32G
# MULTICOLLINEARITY / VIF
# ============================================================

from statsmodels.stats.outliers_influence import (
    variance_inflation_factor
)

vif_records = []

for i, gene in enumerate(
    ORIGINAL_TOP10
):

    vif_value = variance_inflation_factor(
        X_dev_std,
        i
    )

    vif_records.append({
        "Gene": gene,
        "VIF": vif_value
    })

vif_df = pd.DataFrame(
    vif_records
)

vif_df["VIF_gt_5"] = (
    vif_df["VIF"] > 5
)

vif_df["VIF_gt_10"] = (
    vif_df["VIF"] > 10
)

vif_df.to_csv(
    RESULTS_DIR /
    "05_OriginalTop10_VIF.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    vif_df.sort_values(
        "VIF",
        ascending=False
    )
)

In [ ]:
# ============================================================
# CELL 32H
# BOOTSTRAP COEFFICIENT STABILITY
# DEVELOPMENT ONLY
# ============================================================

BOOT_COEF_N = 1000

rng = np.random.default_rng(
    RANDOM_STATE + 70000
)

full_coef = np.asarray(
    original_cox_model.coef_
).reshape(-1)

bootstrap_coef_records = []

n = len(
    development_df
)

for b in range(
    BOOT_COEF_N
):

    idx = rng.integers(
        0,
        n,
        size=n
    )

    X_b = X_dev_std[
        idx,
        :
    ]

    dev_b = development_df.iloc[
        idx
    ].copy()

    try:

        model_b = CoxPHSurvivalAnalysis(
            alpha=0.0
        )

        model_b.fit(
            X_b,
            make_y(dev_b)
        )

        coef_b = np.asarray(
            model_b.coef_
        ).reshape(-1)

        row = {
            "Bootstrap": b + 1
        }

        for gene, value in zip(
            ORIGINAL_TOP10,
            coef_b
        ):

            row[gene] = value

        bootstrap_coef_records.append(
            row
        )

    except Exception:
        continue


bootstrap_coef_df = pd.DataFrame(
    bootstrap_coef_records
)

bootstrap_coef_df.to_csv(
    RESULTS_DIR /
    "06_OriginalTop10_Bootstrap_Coefficients_All.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Valid bootstrap fits:",
    len(bootstrap_coef_df),
    "/",
    BOOT_COEF_N
)

In [ ]:
# ============================================================
# CELL 32I
# BOOTSTRAP COEFFICIENT SUMMARY
# ============================================================

stability_records = []

for i, gene in enumerate(
    ORIGINAL_TOP10
):

    values = (
        bootstrap_coef_df[
            gene
        ]
        .dropna()
        .values
    )

    reference_sign = np.sign(
        full_coef[i]
    )

    sign_stability = np.mean(
        np.sign(values)
        == reference_sign
    )

    lower = np.percentile(
        values,
        2.5
    )

    upper = np.percentile(
        values,
        97.5
    )

    stability_records.append({
        "Gene": gene,
        "Full_Coefficient": full_coef[i],
        "Bootstrap_Median": np.median(values),
        "Bootstrap_Mean": np.mean(values),
        "Bootstrap_SD": np.std(
            values,
            ddof=1
        ),
        "Bootstrap_CI_2.5": lower,
        "Bootstrap_CI_97.5": upper,
        "Sign_Stability": sign_stability,
        "CI_Crosses_Zero":
            (lower <= 0 <= upper)
    })


coef_stability = pd.DataFrame(
    stability_records
)

coef_stability.to_csv(
    RESULTS_DIR /
    "07_OriginalTop10_Bootstrap_Coefficient_Stability.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    coef_stability.sort_values(
        "Sign_Stability",
        ascending=False
    )
)

In [ ]:
# ============================================================
# CELL 32J
# BOOTSTRAP COEFFICIENT STABILITY FIGURE
# ============================================================

plot_df = coef_stability.sort_values(
    "Bootstrap_Median"
).copy()

y = np.arange(
    len(plot_df)
)

x = plot_df[
    "Bootstrap_Median"
].values

lower = (
    x
    - plot_df[
        "Bootstrap_CI_2.5"
    ].values
)

upper = (
    plot_df[
        "Bootstrap_CI_97.5"
    ].values
    - x
)

fig, ax = plt.subplots(
    figsize=(8, 6)
)

ax.errorbar(
    x,
    y,
    xerr=[
        lower,
        upper
    ],
    fmt="o",
    capsize=4
)

ax.axvline(
    0,
    linestyle="--"
)

ax.set_yticks(
    y
)

ax.set_yticklabels(
    plot_df["Gene"]
)

ax.set_xlabel(
    "Standardized Cox coefficient"
)

ax.set_title(
    "Original Top10 Bootstrap Coefficient Stability"
)

fig.tight_layout()

fig.savefig(
    FIGURES_DIR /
    "02_OriginalTop10_Bootstrap_Coefficient_Stability.png",
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    FIGURES_DIR /
    "02_OriginalTop10_Bootstrap_Coefficient_Stability.svg",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# CELL 32K
# LOCK DEVELOPMENT-DERIVED RISK CUTOFF
# ============================================================

DEVELOPMENT_RISK_CUTOFF = float(
    np.median(
        dev_risk
    )
)

development_predictions = pd.DataFrame({
    "Sample_ID":
        development_df[
            MASTER_ID
        ].astype(str).values,

    "OS_time":
        development_df[
            TIME_COL
        ].values,

    "OS":
        development_df[
            EVENT_COL
        ].values,

    "Linear_Predictor":
        dev_risk,

    "Risk_Group":
        np.where(
            dev_risk
            >= DEVELOPMENT_RISK_CUTOFF,
            "High",
            "Low"
        )
})

development_predictions.to_csv(
    RESULTS_DIR /
    "08_OriginalTop10_Development_RiskScores.csv",
    index=False,
    encoding="utf-8-sig"
)

cutoff_df = pd.DataFrame([
    {
        "Model_ID":
            "M01_OriginalTop10_UnpenalizedCox",

        "Cutoff_Method":
            "Development median",

        "Cutoff_Value":
            DEVELOPMENT_RISK_CUTOFF,

        "Derived_From":
            "Development N=238",

        "External_Reoptimization_Allowed":
            "NO"
    }
])

cutoff_df.to_csv(
    RESULTS_DIR /
    "09_OriginalTop10_RiskCutoff_LOCKED.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "LOCKED risk cutoff:",
    DEVELOPMENT_RISK_CUTOFF
)

In [ ]:
# ============================================================
# CELL 32L
# SAVE PREPROCESSING PARAMETERS
# ============================================================

preprocess_table = pd.DataFrame({
    "Feature":
        ORIGINAL_TOP10,

    "Imputation_Median":
        dev_imputer.statistics_,

    "Scaling_Mean":
        dev_scaler.mean_,

    "Scaling_SD":
        dev_scaler.scale_
})

preprocess_table.to_csv(
    RESULTS_DIR /
    "10_OriginalTop10_Preprocessing_Parameters.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    preprocess_table
)

In [ ]:
# ============================================================
# CELL 32M
# SAVE BASELINE SURVIVAL FUNCTION
# ============================================================

baseline_survival = (
    original_cox_model
    .baseline_survival_
)

baseline_survival_df = pd.DataFrame({
    "Time":
        baseline_survival.x,

    "Baseline_Survival":
        baseline_survival.y
})

baseline_survival_df.to_csv(
    RESULTS_DIR /
    "11_OriginalTop10_Baseline_Survival.csv",
    index=False,
    encoding="utf-8-sig"
)

TIME_POINTS = [
    365,
    730,
    1095
]

baseline_timepoints = []

for t in TIME_POINTS:

    baseline_timepoints.append({
        "Time_Days": t,
        "Baseline_Survival":
            float(
                baseline_survival(t)
            )
    })

baseline_timepoints_df = pd.DataFrame(
    baseline_timepoints
)

baseline_timepoints_df.to_csv(
    RESULTS_DIR /
    "12_OriginalTop10_Baseline_Survival_1_2_3Y.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    baseline_timepoints_df
)

In [ ]:
# ============================================================
# CELL 32N
# SAVE DEVELOPMENT MODEL BUNDLE
# ============================================================

MODEL_BUNDLE = {
    "model_id":
        "M01_OriginalTop10_UnpenalizedCox",

    "version":
        "Development_v1.0",

    "features":
        ORIGINAL_TOP10,

    "model":
        original_cox_model,

    "imputer":
        dev_imputer,

    "scaler":
        dev_scaler,

    "risk_cutoff":
        DEVELOPMENT_RISK_CUTOFF,

    "time_unit":
        "days",

    "development_n":
        len(
            development_df
        ),

    "development_events":
        int(
            development_df[
                EVENT_COL
            ].sum()
        ),

    "holdout_used":
        False
}

MODEL_FILE = (
    MODELS_DIR /
    "M01_OriginalTop10_UnpenalizedCox_Development_v1.0.joblib"
)

joblib.dump(
    MODEL_BUNDLE,
    MODEL_FILE
)

print(
    "Model bundle saved:"
)

print(
    MODEL_FILE
)

In [ ]:
# ============================================================
# CELL 32O
# SAVE PORTABLE MODEL SPECIFICATION
# ============================================================

portable_coef = pd.DataFrame({
    "Feature":
        ORIGINAL_TOP10,

    "Coefficient":
        np.asarray(
            original_cox_model.coef_
        ).reshape(-1),

    "Imputation_Median":
        dev_imputer.statistics_,

    "Scaling_Mean":
        dev_scaler.mean_,

    "Scaling_SD":
        dev_scaler.scale_
})

portable_coef.to_csv(
    MODELS_DIR /
    "M01_OriginalTop10_Portable_Model_Coefficients.csv",
    index=False,
    encoding="utf-8-sig"
)

portable_spec = {
    "Model_ID":
        "M01_OriginalTop10_UnpenalizedCox",

    "Version":
        "Development_v1.0",

    "Model_Type":
        "Unpenalized Cox proportional hazards",

    "Features":
        ORIGINAL_TOP10,

    "Transformation":
        "z = (x - DevelopmentMean) / DevelopmentSD",

    "Risk_Formula":
        "Linear predictor = sum(beta_i * z_i)",

    "Missing_Value_Rule":
        "Use Development median",

    "Risk_Group_Cutoff":
        DEVELOPMENT_RISK_CUTOFF,

    "Risk_Group_Cutoff_Rule":
        "Development median; frozen for holdout/external validation",

    "Time_Unit":
        "days",

    "Development_N":
        int(
            len(development_df)
        ),

    "Development_Events":
        int(
            development_df[
                EVENT_COL
            ].sum()
        ),

    "Holdout_N":
        int(
            len(holdout_df)
        ),

    "Holdout_Used":
        False,

    "Feature_ReSelection_In_Holdout":
        False,

    "External_ReTuning":
        False
}

with open(
    MODELS_DIR /
    "M01_OriginalTop10_Portable_Model_Specification.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        portable_spec,
        f,
        ensure_ascii=False,
        indent=2
    )

In [ ]:
# ============================================================
# CELL 32P
# SOFTWARE ENVIRONMENT LOCK
# ============================================================

import sys
import sklearn
import sksurv
import lifelines
import scipy
import matplotlib

environment_info = {
    "Python":
        sys.version,

    "numpy":
        np.__version__,

    "pandas":
        pd.__version__,

    "scikit-learn":
        sklearn.__version__,

    "scikit-survival":
        sksurv.__version__,

    "lifelines":
        lifelines.__version__,

    "scipy":
        scipy.__version__,

    "matplotlib":
        matplotlib.__version__,
}

with open(
    AUDIT_DIR /
    "SOFTWARE_ENVIRONMENT.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        environment_info,
        f,
        indent=2
    )

print(
    json.dumps(
        environment_info,
        indent=2
    )
)

In [ ]:
# ============================================================
# CELL 32Q
# PRE-HOLDOUT MODEL AUDIT
# ============================================================

audit_lines = []

audit_lines.append(
    "TCGA-LIHC PRE-HOLDOUT MODEL AUDIT"
)

audit_lines.append(
    "=" * 80
)

audit_lines.append(
    f"Development N: {len(development_df)}"
)

audit_lines.append(
    f"Development events: {int(development_df[EVENT_COL].sum())}"
)

audit_lines.append(
    f"Holdout N: {len(holdout_df)}"
)

audit_lines.append(
    "Holdout used: NO"
)

audit_lines.append(
    ""
)

audit_lines.append(
    "IMPORTANT CORRECTION:"
)

audit_lines.append(
    "Previous model labelled CoxPH used alpha=0.01 and is therefore Ridge Cox."
)

audit_lines.append(
    "True unpenalized CoxPH (alpha=0) has now been evaluated separately."
)

audit_lines.append(
    ""
)

audit_lines.append(
    "Primary model status: NOT YET FINAL."
)

audit_lines.append(
    "Candidate model definitions are locked before holdout access."
)

audit_lines.append(
    ""
)

audit_lines.append(
    "External validation principle:"
)

audit_lines.append(
    "No external feature selection."
)

audit_lines.append(
    "No external hyperparameter tuning."
)

audit_lines.append(
    "No external cutoff optimization."
)

audit_lines.append(
    "Development preprocessing parameters must be reused."
)

audit_lines.append(
    ""
)

audit_lines.append(
    "PRE-HOLDOUT AUDIT COMPLETE"
)

audit_text = "\n".join(
    audit_lines
)

with open(
    AUDIT_DIR /
    "PRE_HOLDOUT_MODEL_AUDIT.md",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        audit_text
    )

print(
    audit_text
)

In [ ]:
# ============================================================
# CELL 32R
# PH ASSUMPTION MULTIPLE-TESTING CORRECTION
# BH-FDR
# ============================================================

from statsmodels.stats.multitest import multipletests

ph_fdr = ph_table.copy()

# 兼容列名
if "Gene" not in ph_fdr.columns:
    ph_fdr = ph_fdr.rename(
        columns={
            ph_fdr.columns[0]: "Gene"
        }
    )

# BH-FDR
reject, p_adj, _, _ = multipletests(
    ph_fdr["p"].values,
    alpha=0.05,
    method="fdr_bh"
)

ph_fdr["FDR_BH"] = p_adj
ph_fdr["PH_Violation_RawP_lt_0.05"] = (
    ph_fdr["p"] < 0.05
)

ph_fdr["PH_Violation_FDR_lt_0.05"] = reject

ph_fdr = ph_fdr.sort_values(
    "p"
).reset_index(drop=True)

ph_fdr.to_csv(
    RESULTS_DIR /
    "13_OriginalTop10_PH_Assumption_FDR.csv",
    index=False,
    encoding="utf-8-sig"
)

display(ph_fdr)

print("\nRaw P < 0.05:")
display(
    ph_fdr.loc[
        ph_fdr[
            "PH_Violation_RawP_lt_0.05"
        ],
        [
            "Gene",
            "test_statistic",
            "p",
            "FDR_BH"
        ]
    ]
)

print("\nFDR < 0.05:")
display(
    ph_fdr.loc[
        ph_fdr[
            "PH_Violation_FDR_lt_0.05"
        ],
        [
            "Gene",
            "test_statistic",
            "p",
            "FDR_BH"
        ]
    ]
)

In [ ]:
# ============================================================
# CELL 32S
# SCHOENFELD RESIDUALS
# SAVE RAW DATA
# ============================================================

# lifelines CoxPH model:
# cph_inference
#
# fitting data:
# lifelines_df

schoenfeld_resid = (
    cph_inference
    .compute_residuals(
        lifelines_df,
        kind="schoenfeld"
    )
    .copy()
)

# compute_residuals只包含发生事件的样本
event_times = (
    lifelines_df.loc[
        schoenfeld_resid.index,
        "OS_time"
    ]
)

schoenfeld_output = (
    schoenfeld_resid.copy()
)

schoenfeld_output.insert(
    0,
    "OS_time",
    event_times.values
)

schoenfeld_output.insert(
    0,
    "Original_Index",
    schoenfeld_resid.index
)

schoenfeld_output.to_csv(
    RESULTS_DIR /
    "14_OriginalTop10_Schoenfeld_Residuals.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Schoenfeld residual shape:",
    schoenfeld_output.shape
)

display(
    schoenfeld_output.head()
)

In [ ]:
# ============================================================
# CELL 32T
# SCHOENFELD RESIDUAL DIAGNOSTIC FIGURES
# ALL 10 GENES
# ============================================================

from statsmodels.nonparametric.smoothers_lowess import lowess

PH_FIGURE_DIR = (
    FIGURES_DIR /
    "PH_Assumption"
)

PH_FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ph_plot_records = []

for gene in ORIGINAL_TOP10:

    if gene not in schoenfeld_resid.columns:
        print(
            f"Skip {gene}: no Schoenfeld residual"
        )
        continue

    time_values = (
        event_times
        .astype(float)
        .values
    )

    residual_values = (
        schoenfeld_resid[
            gene
        ]
        .astype(float)
        .values
    )

    plot_df = pd.DataFrame({
        "OS_time": time_values,
        "Schoenfeld_Residual":
            residual_values
    }).dropna()

    # LOWESS
    smooth = lowess(
        endog=plot_df[
            "Schoenfeld_Residual"
        ],
        exog=plot_df[
            "OS_time"
        ],
        frac=0.40,
        it=0,
        return_sorted=True
    )

    # PH statistics
    gene_row = (
        ph_fdr.loc[
            ph_fdr["Gene"] == gene
        ]
    )

    if len(gene_row) > 0:

        raw_p = float(
            gene_row.iloc[0]["p"]
        )

        fdr_p = float(
            gene_row.iloc[0]["FDR_BH"]
        )

    else:

        raw_p = np.nan
        fdr_p = np.nan

    # -------------------------------------
    # Plot
    # -------------------------------------

    fig, ax = plt.subplots(
        figsize=(7.5, 5.5)
    )

    ax.scatter(
        plot_df["OS_time"],
        plot_df[
            "Schoenfeld_Residual"
        ],
        alpha=0.55,
        s=24
    )

    ax.plot(
        smooth[:, 0],
        smooth[:, 1],
        linewidth=2
    )

    ax.axhline(
        0,
        linestyle="--",
        linewidth=1
    )

    ax.set_xlabel(
        "Event time (days)"
    )

    ax.set_ylabel(
        "Schoenfeld residual"
    )

    ax.set_title(
        f"{gene}\n"
        f"PH raw P={raw_p:.4g}, "
        f"BH-FDR={fdr_p:.4g}"
    )

    fig.tight_layout()

    png_path = (
        PH_FIGURE_DIR /
        f"PH_Schoenfeld_{gene}.png"
    )

    svg_path = (
        PH_FIGURE_DIR /
        f"PH_Schoenfeld_{gene}.svg"
    )

    fig.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight"
    )

    fig.savefig(
        svg_path,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    ph_plot_records.append({
        "Gene": gene,
        "Raw_P": raw_p,
        "FDR_BH": fdr_p,
        "PNG": str(png_path),
        "SVG": str(svg_path)
    })


ph_plot_manifest = pd.DataFrame(
    ph_plot_records
)

ph_plot_manifest.to_csv(
    RESULTS_DIR /
    "15_OriginalTop10_PH_Figure_Manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

display(ph_plot_manifest)

In [ ]:
# ============================================================
# CELL 32U
# BOOTSTRAP OPTIMISM-CORRECTED C-INDEX
#
# Development only
# Original Top10 + Unpenalized Cox
# ============================================================

BOOT_OPTIMISM_N = 1000

rng = np.random.default_rng(
    RANDOM_STATE + 80000
)

# ============================================================
# Apparent performance
# ============================================================

APPARENT_CINDEX = float(
    cindex_from_risk(
        development_df,
        dev_risk
    )
)

print(
    "Apparent C-index:",
    round(APPARENT_CINDEX, 4)
)

optimism_records = []

n_dev = len(
    development_df
)

for b in range(
    BOOT_OPTIMISM_N
):

    # --------------------------------------------------------
    # Bootstrap sample
    # --------------------------------------------------------

    boot_idx = rng.integers(
        0,
        n_dev,
        size=n_dev
    )

    boot_df = (
        development_df.iloc[
            boot_idx
        ]
        .copy()
        .reset_index(drop=True)
    )

    try:

        # ----------------------------------------------------
        # Fit preprocessing ONLY on bootstrap sample
        # ----------------------------------------------------

        X_boot_raw = (
            boot_df[
                ORIGINAL_TOP10
            ]
            .apply(
                pd.to_numeric,
                errors="coerce"
            )
        )

        boot_imputer = SimpleImputer(
            strategy="median"
        )

        X_boot_imp = (
            boot_imputer
            .fit_transform(
                X_boot_raw
            )
        )

        boot_scaler = StandardScaler()

        X_boot_std = (
            boot_scaler
            .fit_transform(
                X_boot_imp
            )
        )

        # ----------------------------------------------------
        # Fit model on bootstrap sample
        # ----------------------------------------------------

        boot_model = (
            CoxPHSurvivalAnalysis(
                alpha=0.0
            )
        )

        boot_model.fit(
            X_boot_std,
            make_y(
                boot_df
            )
        )

        # ----------------------------------------------------
        # Performance in bootstrap sample
        # ----------------------------------------------------

        boot_risk = (
            boot_model.predict(
                X_boot_std
            )
        )

        c_boot = float(
            cindex_from_risk(
                boot_df,
                boot_risk
            )
        )

        # ----------------------------------------------------
        # Apply bootstrap model to ORIGINAL development cohort
        #
        # IMPORTANT:
        # use bootstrap-fitted imputer + scaler
        # ----------------------------------------------------

        X_original_raw = (
            development_df[
                ORIGINAL_TOP10
            ]
            .apply(
                pd.to_numeric,
                errors="coerce"
            )
        )

        X_original_imp = (
            boot_imputer
            .transform(
                X_original_raw
            )
        )

        X_original_std = (
            boot_scaler
            .transform(
                X_original_imp
            )
        )

        original_risk_from_boot = (
            boot_model.predict(
                X_original_std
            )
        )

        c_original = float(
            cindex_from_risk(
                development_df,
                original_risk_from_boot
            )
        )

        optimism = (
            c_boot
            - c_original
        )

        optimism_records.append({
            "Bootstrap":
                b + 1,

            "C_Index_Bootstrap":
                c_boot,

            "C_Index_Original":
                c_original,

            "Optimism":
                optimism
        })

    except Exception as e:

        continue


optimism_df = pd.DataFrame(
    optimism_records
)

optimism_df.to_csv(
    RESULTS_DIR /
    "16_OriginalTop10_Bootstrap_Optimism_All.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Valid bootstrap iterations:",
    len(optimism_df),
    "/",
    BOOT_OPTIMISM_N
)

In [ ]:
# ============================================================
# CELL 32V
# BOOTSTRAP OPTIMISM SUMMARY
# ============================================================

mean_optimism = float(
    optimism_df[
        "Optimism"
    ].mean()
)

median_optimism = float(
    optimism_df[
        "Optimism"
    ].median()
)

OPTIMISM_CORRECTED_CINDEX = (
    APPARENT_CINDEX
    - mean_optimism
)

optimism_ci_low = float(
    np.percentile(
        optimism_df[
            "Optimism"
        ],
        2.5
    )
)

optimism_ci_high = float(
    np.percentile(
        optimism_df[
            "Optimism"
        ],
        97.5
    )
)

optimism_summary = pd.DataFrame([
    {
        "Model_ID":
            "M01_OriginalTop10_UnpenalizedCox",

        "Development_N":
            len(
                development_df
            ),

        "Events":
            int(
                development_df[
                    EVENT_COL
                ].sum()
            ),

        "Bootstrap_N_Planned":
            BOOT_OPTIMISM_N,

        "Bootstrap_N_Valid":
            len(
                optimism_df
            ),

        "Apparent_C_Index":
            APPARENT_CINDEX,

        "Mean_Optimism":
            mean_optimism,

        "Median_Optimism":
            median_optimism,

        "Optimism_2.5":
            optimism_ci_low,

        "Optimism_97.5":
            optimism_ci_high,

        "Optimism_Corrected_C_Index":
            OPTIMISM_CORRECTED_CINDEX
    }
])

optimism_summary.to_csv(
    RESULTS_DIR /
    "17_OriginalTop10_Bootstrap_Optimism_Summary.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    optimism_summary
)

In [ ]:
# ============================================================
# CELL 32W
# BOOTSTRAP OPTIMISM DISTRIBUTION
# ============================================================

fig, ax = plt.subplots(
    figsize=(8, 5.5)
)

ax.hist(
    optimism_df[
        "Optimism"
    ],
    bins=30,
    alpha=0.8
)

ax.axvline(
    mean_optimism,
    linestyle="--",
    linewidth=2,
    label=(
        f"Mean optimism = "
        f"{mean_optimism:.3f}"
    )
)

ax.axvline(
    0,
    linewidth=1
)

ax.set_xlabel(
    "Bootstrap optimism in C-index"
)

ax.set_ylabel(
    "Frequency"
)

ax.set_title(
    "Original Top10 Cox: Bootstrap Optimism"
)

ax.legend()

fig.tight_layout()

fig.savefig(
    FIGURES_DIR /
    "03_OriginalTop10_Bootstrap_Optimism.png",
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    FIGURES_DIR /
    "03_OriginalTop10_Bootstrap_Optimism.svg",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# CELL 32X
# INTERNAL VALIDATION SUMMARY
# ============================================================

# True unpenalized Cox repeated CV
unpen_cv_row = (
    original_top10_algorithm_summary.loc[
        original_top10_algorithm_summary[
            "Model"
        ]
        == "OriginalTop10_UnpenalizedCox"
    ]
    .iloc[0]
)

internal_validation_summary = pd.DataFrame([
    {
        "Validation_Method":
            "Apparent Development",

        "C_Index":
            APPARENT_CINDEX,

        "SD":
            np.nan,

        "N":
            len(
                development_df
            )
    },
    {
        "Validation_Method":
            "Repeated 5x5 CV",

        "C_Index":
            float(
                unpen_cv_row[
                    "Mean_C_Index"
                ]
            ),

        "SD":
            float(
                unpen_cv_row[
                    "SD_C_Index"
                ]
            ),

        "N":
            len(
                development_df
            )
    },
    {
        "Validation_Method":
            "Bootstrap optimism-corrected",

        "C_Index":
            OPTIMISM_CORRECTED_CINDEX,

        "SD":
            np.nan,

        "N":
            len(
                development_df
            )
    }
])

internal_validation_summary.to_csv(
    RESULTS_DIR /
    "18_OriginalTop10_Internal_Validation_Summary.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    internal_validation_summary
)

In [ ]:
# ============================================================
# CELL 32Y
# FINAL PRE-HOLDOUT STATUS
# ============================================================

n_raw_ph = int(
    ph_fdr[
        "PH_Violation_RawP_lt_0.05"
    ].sum()
)

n_fdr_ph = int(
    ph_fdr[
        "PH_Violation_FDR_lt_0.05"
    ].sum()
)

final_audit_lines = []

final_audit_lines.append(
    "TCGA-LIHC FINAL PRE-HOLDOUT AUDIT"
)

final_audit_lines.append(
    "=" * 80
)

final_audit_lines.append(
    f"Development N: {len(development_df)}"
)

final_audit_lines.append(
    f"Development events: "
    f"{int(development_df[EVENT_COL].sum())}"
)

final_audit_lines.append(
    f"Holdout N: {len(holdout_df)}"
)

final_audit_lines.append(
    "Holdout used: NO"
)

final_audit_lines.append(
    ""
)

final_audit_lines.append(
    "PRIMARY CANDIDATE:"
)

final_audit_lines.append(
    "Original Top10 + Unpenalized CoxPH"
)

final_audit_lines.append(
    ""
)

final_audit_lines.append(
    f"Repeated CV C-index: "
    f"{float(unpen_cv_row['Mean_C_Index']):.4f}"
)

final_audit_lines.append(
    f"Repeated CV SD: "
    f"{float(unpen_cv_row['SD_C_Index']):.4f}"
)

final_audit_lines.append(
    f"Apparent C-index: "
    f"{APPARENT_CINDEX:.4f}"
)

final_audit_lines.append(
    f"Bootstrap mean optimism: "
    f"{mean_optimism:.4f}"
)

final_audit_lines.append(
    f"Bootstrap optimism-corrected C-index: "
    f"{OPTIMISM_CORRECTED_CINDEX:.4f}"
)

final_audit_lines.append(
    ""
)

final_audit_lines.append(
    f"PH raw P<0.05 genes: {n_raw_ph}"
)

final_audit_lines.append(
    f"PH FDR<0.05 genes: {n_fdr_ph}"
)

final_audit_lines.append(
    ""
)

final_audit_lines.append(
    "Risk cutoff:"
)

final_audit_lines.append(
    f"{DEVELOPMENT_RISK_CUTOFF}"
)

final_audit_lines.append(
    "Cutoff source: Development median"
)

final_audit_lines.append(
    "Holdout cutoff reoptimization allowed: NO"
)

final_audit_lines.append(
    "External cutoff reoptimization allowed: NO"
)

final_audit_lines.append(
    ""
)

final_audit_lines.append(
    "Model preprocessing:"
)

final_audit_lines.append(
    "Development median imputation"
)

final_audit_lines.append(
    "Development mean/SD standardization"
)

final_audit_lines.append(
    "External/holdout preprocessing re-estimation: NO"
)

final_audit_lines.append(
    ""
)

final_audit_lines.append(
    "PRE-HOLDOUT DEVELOPMENT ANALYSIS COMPLETE"
)

final_audit_lines.append(
    "HOLDOUT STATUS: LOCKED"
)

final_audit_text = "\n".join(
    final_audit_lines
)

with open(
    AUDIT_DIR /
    "FINAL_PRE_HOLDOUT_AUDIT.md",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        final_audit_text
    )

print(
    final_audit_text
)

In [ ]:
# ============================================================
# CELL 33
# PHASE B DIRECTORY STRUCTURE
# ============================================================

HOLDOUT_RESULTS_DIR = (
    RESULTS_DIR /
    "Holdout_Validation"
)

HOLDOUT_FIGURES_DIR = (
    FIGURES_DIR /
    "Holdout_Validation"
)

FROZEN_MODELS_DIR = (
    MODELS_DIR /
    "Frozen_PreHoldout"
)

for d in [
    HOLDOUT_RESULTS_DIR,
    HOLDOUT_FIGURES_DIR,
    FROZEN_MODELS_DIR
]:
    d.mkdir(
        parents=True,
        exist_ok=True
    )

print("Holdout Results:", HOLDOUT_RESULTS_DIR)
print("Holdout Figures:", HOLDOUT_FIGURES_DIR)
print("Frozen Models:", FROZEN_MODELS_DIR)

In [ ]:
# ============================================================
# CELL 34
# FREEZE ALL PRE-SPECIFIED MODELS
# DEVELOPMENT ONLY
# ============================================================

import joblib
import hashlib

assert len(development_df) == 238
assert len(holdout_df) == 102

assert len(ORIGINAL_TOP10) == 10
assert len(STABLE_GENES10) == 10
assert len(FINAL_DYNAMIC_GENES) == 9
assert len(GENES10_BIO7) == 17


def fit_frozen_cox_bundle(
    train_df,
    features,
    model_id,
    alpha=0.0,
    role="Secondary"
):
    X_raw = (
        train_df[features]
        .apply(
            pd.to_numeric,
            errors="coerce"
        )
    )

    imputer = SimpleImputer(
        strategy="median"
    )

    X_imp = imputer.fit_transform(
        X_raw
    )

    scaler = StandardScaler()

    X_std = scaler.fit_transform(
        X_imp
    )

    model = CoxPHSurvivalAnalysis(
        alpha=alpha
    )

    model.fit(
        X_std,
        make_y(train_df)
    )

    bundle = {
        "Model_ID": model_id,
        "Role": role,
        "Algorithm": (
            "Unpenalized CoxPH"
            if alpha == 0
            else f"Ridge Cox alpha={alpha}"
        ),
        "Features": list(features),
        "Imputer": imputer,
        "Scaler": scaler,
        "Model": model,
        "Alpha": alpha,
        "L1_Ratio": None,
        "Development_N": len(train_df),
        "Development_Events":
            int(train_df[EVENT_COL].sum()),
        "Holdout_Used": False
    }

    return bundle


def fit_frozen_elasticnet_bundle(
    train_df,
    features,
    model_id,
    random_state,
    role="Secondary"
):
    # ---------------------------------------------
    # Tune alpha using DEVELOPMENT only
    # ---------------------------------------------
    best_alpha = tune_elastic_alpha(
        train_df,
        features,
        random_state
    )

    X_raw = (
        train_df[features]
        .apply(
            pd.to_numeric,
            errors="coerce"
        )
    )

    imputer = SimpleImputer(
        strategy="median"
    )

    X_imp = imputer.fit_transform(
        X_raw
    )

    scaler = StandardScaler()

    X_std = scaler.fit_transform(
        X_imp
    )

    model = CoxnetSurvivalAnalysis(
        l1_ratio=L1_RATIO,
        alphas=[best_alpha],
        max_iter=100000,
        fit_baseline_model=True
    )

    model.fit(
        X_std,
        make_y(train_df)
    )

    bundle = {
        "Model_ID": model_id,
        "Role": role,
        "Algorithm": "ElasticNet Cox",
        "Features": list(features),
        "Imputer": imputer,
        "Scaler": scaler,
        "Model": model,
        "Alpha": float(best_alpha),
        "L1_Ratio": L1_RATIO,
        "Development_N": len(train_df),
        "Development_Events":
            int(train_df[EVENT_COL].sum()),
        "Holdout_Used": False
    }

    return bundle

In [ ]:
# ============================================================
# CELL 35
# BUILD FROZEN MODEL BUNDLES
# ============================================================

FROZEN_MODELS = {}

# ------------------------------------------------------------
# Primary
# ------------------------------------------------------------

FROZEN_MODELS[
    "M01_OriginalTop10_UnpenalizedCox"
] = fit_frozen_cox_bundle(
    development_df,
    ORIGINAL_TOP10,
    model_id="M01_OriginalTop10_UnpenalizedCox",
    alpha=0.0,
    role="PRIMARY"
)

# ------------------------------------------------------------
# Original Top10 ElasticNet
# ------------------------------------------------------------

FROZEN_MODELS[
    "M02_OriginalTop10_ElasticNet"
] = fit_frozen_elasticnet_bundle(
    development_df,
    ORIGINAL_TOP10,
    model_id="M02_OriginalTop10_ElasticNet",
    random_state=RANDOM_STATE + 101,
    role="Sensitivity"
)

# ------------------------------------------------------------
# StableGenes10
# ------------------------------------------------------------

FROZEN_MODELS[
    "M03_StableGenes10_ElasticNet"
] = fit_frozen_elasticnet_bundle(
    development_df,
    STABLE_GENES10,
    model_id="M03_StableGenes10_ElasticNet",
    random_state=RANDOM_STATE + 102,
    role="Sensitivity"
)

# ------------------------------------------------------------
# Dynamic 9 - Cox
# ------------------------------------------------------------

FROZEN_MODELS[
    "M04_Dynamic9_UnpenalizedCox"
] = fit_frozen_cox_bundle(
    development_df,
    FINAL_DYNAMIC_GENES,
    model_id="M04_Dynamic9_UnpenalizedCox",
    alpha=0.0,
    role="Exploratory"
)

# ------------------------------------------------------------
# Dynamic 9 - ElasticNet
# ------------------------------------------------------------

FROZEN_MODELS[
    "M05_Dynamic9_ElasticNet"
] = fit_frozen_elasticnet_bundle(
    development_df,
    FINAL_DYNAMIC_GENES,
    model_id="M05_Dynamic9_ElasticNet",
    random_state=RANDOM_STATE + 103,
    role="Exploratory"
)

# ------------------------------------------------------------
# Genes10 + Bio7
# ------------------------------------------------------------

FROZEN_MODELS[
    "M06_Genes10Bio7_ElasticNet"
] = fit_frozen_elasticnet_bundle(
    development_df,
    GENES10_BIO7,
    model_id="M06_Genes10Bio7_ElasticNet",
    random_state=RANDOM_STATE + 104,
    role="Incremental"
)

# ------------------------------------------------------------
# Clinical benchmark
# ------------------------------------------------------------

FROZEN_MODELS[
    "M07_Clinical_ElasticNet"
] = fit_frozen_elasticnet_bundle(
    development_df,
    CLINICAL_FEATURES,
    model_id="M07_Clinical_ElasticNet",
    random_state=RANDOM_STATE + 105,
    role="Clinical benchmark"
)

print(
    "Frozen models:",
    len(FROZEN_MODELS)
)

In [ ]:
# ============================================================
# CELL 36
# SAVE FROZEN MODELS
# ============================================================

frozen_model_records = []

for model_id, bundle in FROZEN_MODELS.items():

    model_path = (
        FROZEN_MODELS_DIR /
        f"{model_id}_DevelopmentFrozen.joblib"
    )

    joblib.dump(
        bundle,
        model_path
    )

    model = bundle["Model"]

    coef = np.asarray(
        model.coef_
    ).reshape(-1)

    portable_table = pd.DataFrame({
        "Feature":
            bundle["Features"],

        "Coefficient":
            coef,

        "Imputation_Median":
            bundle["Imputer"].statistics_,

        "Scaling_Mean":
            bundle["Scaler"].mean_,

        "Scaling_SD":
            bundle["Scaler"].scale_
    })

    portable_table.to_csv(
        FROZEN_MODELS_DIR /
        f"{model_id}_PortableParameters.csv",
        index=False,
        encoding="utf-8-sig"
    )

    frozen_model_records.append({
        "Model_ID":
            model_id,

        "Role":
            bundle["Role"],

        "Algorithm":
            bundle["Algorithm"],

        "N_Features":
            len(bundle["Features"]),

        "Features":
            ";".join(
                bundle["Features"]
            ),

        "Alpha":
            bundle["Alpha"],

        "L1_Ratio":
            bundle["L1_Ratio"],

        "Development_N":
            bundle["Development_N"],

        "Development_Events":
            bundle["Development_Events"],

        "Holdout_Used":
            "NO",

        "Model_File":
            str(model_path)
    })


FROZEN_MODEL_REGISTRY = pd.DataFrame(
    frozen_model_records
)

FROZEN_MODEL_REGISTRY.to_csv(
    HOLDOUT_RESULTS_DIR /
    "00_PRE_HOLDOUT_FROZEN_MODEL_REGISTRY.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    FROZEN_MODEL_REGISTRY
)

In [ ]:
# ============================================================
# CELL 37
# OFFICIAL HOLDOUT UNLOCK
# ============================================================

UNLOCK_HOLDOUT = True

if not UNLOCK_HOLDOUT:
    raise RuntimeError(
        "Holdout remains locked."
    )

HOLDOUT_UNLOCK_TIME = (
    datetime.now()
    .strftime("%Y-%m-%d %H:%M:%S")
)

unlock_log = {
    "Holdout_Unlocked":
        True,

    "Unlock_Time":
        HOLDOUT_UNLOCK_TIME,

    "Development_N":
        len(development_df),

    "Holdout_N":
        len(holdout_df),

    "Primary_Model":
        "M01_OriginalTop10_UnpenalizedCox",

    "Feature_Reselection_After_Unlock":
        "PROHIBITED",

    "Hyperparameter_Retuning_After_Unlock":
        "PROHIBITED",

    "Risk_Cutoff_Reoptimization":
        "PROHIBITED",

    "Primary_Model_Change_Based_On_Holdout":
        "PROHIBITED"
}

with open(
    AUDIT_DIR /
    "HOLDOUT_UNLOCK_LOG.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        unlock_log,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    json.dumps(
        unlock_log,
        ensure_ascii=False,
        indent=2
    )
)

In [ ]:
# ============================================================
# CELL 37A
# LOCK PRIMARY MODEL ID
# ============================================================

PRIMARY_MODEL_ID = "M01_OriginalTop10_UnpenalizedCox"

if PRIMARY_MODEL_ID not in FROZEN_MODELS:
    raise KeyError(
        f"Primary model not found in FROZEN_MODELS: "
        f"{PRIMARY_MODEL_ID}"
    )

primary_bundle = FROZEN_MODELS[
    PRIMARY_MODEL_ID
]

print("=" * 70)
print("PRIMARY MODEL LOCKED")
print("=" * 70)
print("Model ID :", PRIMARY_MODEL_ID)
print("Role     :", primary_bundle["Role"])
print("Algorithm:", primary_bundle["Algorithm"])
print("Features :", len(primary_bundle["Features"]))
print(primary_bundle["Features"])

In [ ]:
# ============================================================
# CELL 38
# APPLY FROZEN MODELS TO HOLDOUT
# ============================================================

def transform_with_bundle(
    bundle,
    df
):
    features = bundle[
        "Features"
    ]

    X_raw = (
        df[features]
        .apply(
            pd.to_numeric,
            errors="coerce"
        )
    )

    X_imp = (
        bundle["Imputer"]
        .transform(
            X_raw
        )
    )

    X_std = (
        bundle["Scaler"]
        .transform(
            X_imp
        )
    )

    return X_std


def predict_frozen_bundle(
    bundle,
    df
):
    X = transform_with_bundle(
        bundle,
        df
    )

    return np.asarray(
        bundle["Model"]
        .predict(X)
    ).reshape(-1)


prediction_table = pd.DataFrame({
    "Sample_ID":
        holdout_df[
            MASTER_ID
        ].astype(str).values,

    "OS_time":
        holdout_df[
            TIME_COL
        ].astype(float).values,

    "OS":
        holdout_df[
            EVENT_COL
        ].astype(int).values
})


for model_id, bundle in FROZEN_MODELS.items():

    prediction_table[
        model_id
    ] = predict_frozen_bundle(
        bundle,
        holdout_df
    )


prediction_table.to_csv(
    HOLDOUT_RESULTS_DIR /
    "01_Holdout_All_Model_Predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Holdout prediction table:",
    prediction_table.shape
)

display(
    prediction_table.head()
)

In [ ]:
# ============================================================
# CELL 39
# HOLDOUT C-INDEX + 2000 BOOTSTRAP CI
# ============================================================

HOLDOUT_BOOTSTRAP_N = 2000


def stable_seed(text_value):
    digest = hashlib.sha256(
        text_value.encode("utf-8")
    ).hexdigest()

    return int(
        digest[:8],
        16
    )


holdout_times = (
    holdout_df[
        TIME_COL
    ].astype(float).values
)

holdout_events = (
    holdout_df[
        EVENT_COL
    ].astype(int).values
)


cindex_summary_records = []
cindex_boot_records = []


for model_id in FROZEN_MODELS.keys():

    risk = (
        prediction_table[
            model_id
        ].values
    )

    point_c = float(
        concordance_index_censored(
            holdout_events.astype(bool),
            holdout_times,
            risk
        )[0]
    )

    rng = np.random.default_rng(
        stable_seed(
            model_id +
            "_CINDEX_BOOT"
        )
    )

    boot_values = []

    for b in range(
        HOLDOUT_BOOTSTRAP_N
    ):

        idx = rng.integers(
            0,
            len(holdout_df),
            size=len(holdout_df)
        )

        try:

            c = float(
                concordance_index_censored(
                    holdout_events[
                        idx
                    ].astype(bool),

                    holdout_times[
                        idx
                    ],

                    risk[
                        idx
                    ]
                )[0]
            )

            boot_values.append(
                c
            )

            cindex_boot_records.append({
                "Model_ID":
                    model_id,

                "Bootstrap":
                    b + 1,

                "C_Index":
                    c
            })

        except Exception:
            continue

    boot_values = np.asarray(
        boot_values
    )

    cindex_summary_records.append({
        "Model_ID":
            model_id,

        "Role":
            FROZEN_MODELS[
                model_id
            ]["Role"],

        "Point_C_Index":
            point_c,

        "Bootstrap_Mean":
            np.mean(
                boot_values
            ),

        "CI_2.5":
            np.percentile(
                boot_values,
                2.5
            ),

        "CI_97.5":
            np.percentile(
                boot_values,
                97.5
            ),

        "Bootstrap_N_Valid":
            len(
                boot_values
            )
    })


holdout_cindex = pd.DataFrame(
    cindex_summary_records
)

holdout_cindex_boot = pd.DataFrame(
    cindex_boot_records
)

holdout_cindex.to_csv(
    HOLDOUT_RESULTS_DIR /
    "02_Holdout_CIndex_Bootstrap_Summary.csv",
    index=False,
    encoding="utf-8-sig"
)

holdout_cindex_boot.to_csv(
    HOLDOUT_RESULTS_DIR /
    "03_Holdout_CIndex_Bootstrap_All.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    holdout_cindex.sort_values(
        "Point_C_Index",
        ascending=False
    )
)

In [ ]:
# ============================================================
# CELL 40
# HOLDOUT C-INDEX FOREST PLOT
# ============================================================

plot_df = (
    holdout_cindex
    .sort_values(
        "Point_C_Index"
    )
    .copy()
)

y = np.arange(
    len(plot_df)
)

x = (
    plot_df[
        "Point_C_Index"
    ].values
)

low = (
    x
    - plot_df[
        "CI_2.5"
    ].values
)

high = (
    plot_df[
        "CI_97.5"
    ].values
    - x
)

fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.errorbar(
    x,
    y,
    xerr=[
        low,
        high
    ],
    fmt="o",
    capsize=4
)

ax.axvline(
    0.5,
    linestyle="--",
    linewidth=1
)

ax.set_yticks(y)

ax.set_yticklabels(
    plot_df[
        "Model_ID"
    ]
)

ax.set_xlabel(
    "Holdout C-index (95% bootstrap CI)"
)

ax.set_title(
    "Holdout Validation: Model Discrimination"
)

fig.tight_layout()

fig.savefig(
    HOLDOUT_FIGURES_DIR /
    "01_Holdout_CIndex_Forest.png",
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    HOLDOUT_FIGURES_DIR /
    "01_Holdout_CIndex_Forest.svg",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# CELL 41
# PAIRED BOOTSTRAP DELTA C-INDEX
# VS PRIMARY
# ============================================================

PRIMARY_MODEL_ID = (
    "M01_OriginalTop10_UnpenalizedCox"
)

primary_risk = (
    prediction_table[
        PRIMARY_MODEL_ID
    ].values
)

delta_records = []
delta_all_records = []


for model_id in FROZEN_MODELS.keys():

    if model_id == PRIMARY_MODEL_ID:
        continue

    model_risk = (
        prediction_table[
            model_id
        ].values
    )

    primary_c = float(
        concordance_index_censored(
            holdout_events.astype(bool),
            holdout_times,
            primary_risk
        )[0]
    )

    model_c = float(
        concordance_index_censored(
            holdout_events.astype(bool),
            holdout_times,
            model_risk
        )[0]
    )

    observed_delta = (
        model_c
        - primary_c
    )

    rng = np.random.default_rng(
        stable_seed(
            model_id +
            "_PAIRED_DELTA"
        )
    )

    deltas = []

    for b in range(
        HOLDOUT_BOOTSTRAP_N
    ):

        idx = rng.integers(
            0,
            len(holdout_df),
            size=len(holdout_df)
        )

        try:

            c_primary = float(
                concordance_index_censored(
                    holdout_events[
                        idx
                    ].astype(bool),

                    holdout_times[
                        idx
                    ],

                    primary_risk[
                        idx
                    ]
                )[0]
            )

            c_model = float(
                concordance_index_censored(
                    holdout_events[
                        idx
                    ].astype(bool),

                    holdout_times[
                        idx
                    ],

                    model_risk[
                        idx
                    ]
                )[0]
            )

            delta = (
                c_model
                - c_primary
            )

            deltas.append(
                delta
            )

            delta_all_records.append({
                "Compared_Model":
                    model_id,

                "Bootstrap":
                    b + 1,

                "Delta_C_Index":
                    delta
            })

        except Exception:
            continue

    deltas = np.asarray(
        deltas
    )

    delta_records.append({
        "Primary_Model":
            PRIMARY_MODEL_ID,

        "Compared_Model":
            model_id,

        "Observed_Delta_C":
            observed_delta,

        "Bootstrap_Mean_Delta":
            np.mean(
                deltas
            ),

        "Delta_CI_2.5":
            np.percentile(
                deltas,
                2.5
            ),

        "Delta_CI_97.5":
            np.percentile(
                deltas,
                97.5
            )
    })


delta_summary = pd.DataFrame(
    delta_records
)

delta_all = pd.DataFrame(
    delta_all_records
)

delta_summary.to_csv(
    HOLDOUT_RESULTS_DIR /
    "04_Holdout_Paired_Delta_CIndex.csv",
    index=False,
    encoding="utf-8-sig"
)

delta_all.to_csv(
    HOLDOUT_RESULTS_DIR /
    "05_Holdout_Paired_Delta_CIndex_AllBootstrap.csv",
    index=False,
    encoding="utf-8-sig"
)

display(delta_summary)

In [ ]:
# ============================================================
# CELL 42
# TIME-DEPENDENT AUC PREPARATION
# ============================================================

from sksurv.metrics import (
    cumulative_dynamic_auc,
    brier_score
)

REQUESTED_TIME_POINTS = np.array([
    365.0,
    730.0,
    1095.0
])

max_dev_time = float(
    development_df[
        TIME_COL
    ].max()
)

tau = (
    max_dev_time
    - 1e-6
)

VALID_TIME_POINTS = (
    REQUESTED_TIME_POINTS[
        REQUESTED_TIME_POINTS
        < tau
    ]
)

print(
    "Development max follow-up:",
    max_dev_time
)

print(
    "Valid AUC time points:",
    VALID_TIME_POINTS
)

if len(
    VALID_TIME_POINTS
) == 0:

    raise RuntimeError(
        "No requested AUC time point lies within "
        "Development follow-up support."
    )


# ------------------------------------------------------------
# Administrative truncation only beyond Development support
# This does NOT change status before evaluation horizons.
# ------------------------------------------------------------

holdout_ipcw = (
    holdout_df.copy()
)

mask_late = (
    holdout_ipcw[
        TIME_COL
    ].astype(float)
    >= tau
)

holdout_ipcw.loc[
    mask_late,
    TIME_COL
] = tau

holdout_ipcw.loc[
    mask_late,
    EVENT_COL
] = 0

y_dev_ipcw = make_y(
    development_df
)

y_holdout_ipcw = make_y(
    holdout_ipcw
)

print(
    "Administratively truncated holdout:",
    int(mask_late.sum())
)

In [ ]:
# ============================================================
# CELL 43
# TIME-DEPENDENT AUC + BOOTSTRAP CI
# ============================================================

AUC_BOOTSTRAP_N = 1000

auc_records = []
auc_boot_records = []


for model_id in FROZEN_MODELS.keys():

    risk = (
        prediction_table[
            model_id
        ].values
    )

    auc_values, mean_auc = (
        cumulative_dynamic_auc(
            y_dev_ipcw,
            y_holdout_ipcw,
            risk,
            VALID_TIME_POINTS
        )
    )

    # point estimates
    point_auc_map = {
        float(t): float(a)
        for t, a in zip(
            VALID_TIME_POINTS,
            auc_values
        )
    }

    rng = np.random.default_rng(
        stable_seed(
            model_id +
            "_AUC_BOOT"
        )
    )

    bootstrap_by_time = {
        float(t): []
        for t in VALID_TIME_POINTS
    }

    for b in range(
        AUC_BOOTSTRAP_N
    ):

        idx = rng.integers(
            0,
            len(holdout_ipcw),
            size=len(holdout_ipcw)
        )

        try:

            y_boot = (
                y_holdout_ipcw[
                    idx
                ]
            )

            risk_boot = (
                risk[
                    idx
                ]
            )

            auc_boot, _ = (
                cumulative_dynamic_auc(
                    y_dev_ipcw,
                    y_boot,
                    risk_boot,
                    VALID_TIME_POINTS
                )
            )

            for t, a in zip(
                VALID_TIME_POINTS,
                auc_boot
            ):

                bootstrap_by_time[
                    float(t)
                ].append(
                    float(a)
                )

                auc_boot_records.append({
                    "Model_ID":
                        model_id,

                    "Bootstrap":
                        b + 1,

                    "Time_Days":
                        float(t),

                    "AUC":
                        float(a)
                })

        except Exception:
            continue


    for t in VALID_TIME_POINTS:

        t = float(t)

        vals = np.asarray(
            bootstrap_by_time[
                t
            ]
        )

        auc_records.append({
            "Model_ID":
                model_id,

            "Role":
                FROZEN_MODELS[
                    model_id
                ]["Role"],

            "Time_Days":
                t,

            "AUC":
                point_auc_map[t],

            "CI_2.5":
                np.percentile(
                    vals,
                    2.5
                ),

            "CI_97.5":
                np.percentile(
                    vals,
                    97.5
                ),

            "Bootstrap_N_Valid":
                len(vals)
        })


holdout_auc = pd.DataFrame(
    auc_records
)

holdout_auc_boot = pd.DataFrame(
    auc_boot_records
)

holdout_auc.to_csv(
    HOLDOUT_RESULTS_DIR /
    "06_Holdout_TimeDependent_AUC.csv",
    index=False,
    encoding="utf-8-sig"
)

holdout_auc_boot.to_csv(
    HOLDOUT_RESULTS_DIR /
    "07_Holdout_TimeDependent_AUC_AllBootstrap.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    holdout_auc
)

In [ ]:
# ============================================================
# CELL 44
# TIME-DEPENDENT AUC FIGURE
# ============================================================

fig, ax = plt.subplots(
    figsize=(9, 6)
)

for model_id in FROZEN_MODELS.keys():

    tmp = (
        holdout_auc[
            holdout_auc[
                "Model_ID"
            ] == model_id
        ]
        .sort_values(
            "Time_Days"
        )
    )

    ax.plot(
        tmp[
            "Time_Days"
        ] / 365.0,

        tmp[
            "AUC"
        ],

        marker="o",
        label=model_id
    )


ax.axhline(
    0.5,
    linestyle="--",
    linewidth=1
)

ax.set_xlabel(
    "Time (years)"
)

ax.set_ylabel(
    "Time-dependent AUC"
)

ax.set_ylim(
    0.4,
    1.0
)

ax.set_title(
    "Holdout Time-dependent AUC"
)

ax.legend(
    fontsize=7,
    loc="best"
)

fig.tight_layout()

fig.savefig(
    HOLDOUT_FIGURES_DIR /
    "02_Holdout_TimeDependent_AUC.png",
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    HOLDOUT_FIGURES_DIR /
    "02_Holdout_TimeDependent_AUC.svg",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# CELL 45
# PRIMARY MODEL ABSOLUTE SURVIVAL PROBABILITIES
# ============================================================

# ------------------------------------------------------------
# Safety check
# ------------------------------------------------------------

if "PRIMARY_MODEL_ID" not in globals():
    PRIMARY_MODEL_ID = "M01_OriginalTop10_UnpenalizedCox"

if PRIMARY_MODEL_ID not in FROZEN_MODELS:
    raise KeyError(
        f"Primary model {PRIMARY_MODEL_ID} "
        f"not found in FROZEN_MODELS."
    )

primary_bundle = FROZEN_MODELS[
    PRIMARY_MODEL_ID
]

print(
    "Using primary model:",
    PRIMARY_MODEL_ID
)

# ------------------------------------------------------------
# Transform Holdout using DEVELOPMENT-fitted preprocessing
# ------------------------------------------------------------

X_holdout_primary = transform_with_bundle(
    primary_bundle,
    holdout_df
)

# ------------------------------------------------------------
# Predict survival functions
# ------------------------------------------------------------

primary_surv_functions = (
    primary_bundle[
        "Model"
    ]
    .predict_survival_function(
        X_holdout_primary
    )
)

# ------------------------------------------------------------
# Survival probability matrix
# ------------------------------------------------------------

primary_survival_matrix = np.zeros(
    (
        len(holdout_df),
        len(VALID_TIME_POINTS)
    )
)

for i, fn in enumerate(
    primary_surv_functions
):

    for j, t in enumerate(
        VALID_TIME_POINTS
    ):

        primary_survival_matrix[
            i,
            j
        ] = float(
            fn(t)
        )

# ------------------------------------------------------------
# Event probability = 1 - survival probability
# ------------------------------------------------------------

primary_event_matrix = (
    1.0
    - primary_survival_matrix
)

# ------------------------------------------------------------
# Add predictions to Holdout prediction table
# ------------------------------------------------------------

for j, t in enumerate(
    VALID_TIME_POINTS
):

    prediction_table[
        f"Primary_SurvivalProb_{int(t)}d"
    ] = (
        primary_survival_matrix[
            :,
            j
        ]
    )

    prediction_table[
        f"Primary_EventProb_{int(t)}d"
    ] = (
        primary_event_matrix[
            :,
            j
        ]
    )

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

prediction_table.to_csv(
    HOLDOUT_RESULTS_DIR /
    "08_Holdout_Predictions_With_Primary_EventProbability.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Primary survival probability matrix:",
    primary_survival_matrix.shape
)

print(
    "Primary event probability matrix:",
    primary_event_matrix.shape
)

print(
    "\nSaved:"
)

print(
    HOLDOUT_RESULTS_DIR /
    "08_Holdout_Predictions_With_Primary_EventProbability.csv"
)

display(
    prediction_table.head()
)

In [ ]:
# ============================================================
# CELL 46
# PRIMARY CALIBRATION
# KM-BASED OBSERVED EVENT PROBABILITY
# ============================================================

from lifelines import (
    KaplanMeierFitter
)

CALIBRATION_BINS = 5

calibration_records = []
brier_records = []


for j, t in enumerate(
    VALID_TIME_POINTS
):

    t = float(t)

    pred_event = (
        primary_event_matrix[
            :,
            j
        ]
    )

    tmp = pd.DataFrame({
        "OS_time":
            holdout_df[
                TIME_COL
            ].astype(float).values,

        "OS":
            holdout_df[
                EVENT_COL
            ].astype(int).values,

        "Predicted_Event_Probability":
            pred_event
    })

    # Quantile bins
    tmp["Calibration_Bin"] = (
        pd.qcut(
            tmp[
                "Predicted_Event_Probability"
            ],
            q=CALIBRATION_BINS,
            labels=False,
            duplicates="drop"
        )
    )

    for bin_id, g in tmp.groupby(
        "Calibration_Bin"
    ):

        kmf = KaplanMeierFitter()

        kmf.fit(
            durations=g[
                "OS_time"
            ],
            event_observed=g[
                "OS"
            ]
        )

        observed_event = (
            1
            - float(
                kmf.predict(
                    t
                )
            )
        )

        calibration_records.append({
            "Time_Days":
                t,

            "Calibration_Bin":
                int(bin_id),

            "N":
                len(g),

            "Mean_Predicted_Event_Probability":
                g[
                    "Predicted_Event_Probability"
                ].mean(),

            "Observed_Event_Probability_KM":
                observed_event
        })

    # --------------------------------------------------------
    # IPCW Brier score
    # --------------------------------------------------------

    _, brier_values = (
        brier_score(
            y_dev_ipcw,
            y_holdout_ipcw,
            primary_survival_matrix[
                :,
                [j]
            ],
            np.array([
                t
            ])
        )
    )

    brier_records.append({
        "Time_Days":
            t,

        "Brier_Score":
            float(
                brier_values[0]
            )
    })


calibration_df = pd.DataFrame(
    calibration_records
)

brier_df = pd.DataFrame(
    brier_records
)

calibration_df.to_csv(
    HOLDOUT_RESULTS_DIR /
    "09_Primary_Holdout_Calibration_Table.csv",
    index=False,
    encoding="utf-8-sig"
)

brier_df.to_csv(
    HOLDOUT_RESULTS_DIR /
    "10_Primary_Holdout_Brier_Score.csv",
    index=False,
    encoding="utf-8-sig"
)

display(calibration_df)
display(brier_df)

In [ ]:
# ============================================================
# CELL 47
# CALIBRATION FIGURES
# ============================================================

for t in VALID_TIME_POINTS:

    t = float(t)

    plot_df = (
        calibration_df[
            calibration_df[
                "Time_Days"
            ] == t
        ]
        .sort_values(
            "Mean_Predicted_Event_Probability"
        )
    )

    fig, ax = plt.subplots(
        figsize=(6, 6)
    )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1,
        label="Ideal"
    )

    ax.plot(
        plot_df[
            "Mean_Predicted_Event_Probability"
        ],
        plot_df[
            "Observed_Event_Probability_KM"
        ],
        marker="o",
        linewidth=2,
        label="Primary model"
    )

    ax.set_xlim(
        0,
        1
    )

    ax.set_ylim(
        0,
        1
    )

    ax.set_xlabel(
        "Predicted event probability"
    )

    ax.set_ylabel(
        "Observed event probability (KM)"
    )

    ax.set_title(
        f"Holdout Calibration - "
        f"{t / 365:.0f} Year"
    )

    ax.legend()

    fig.tight_layout()

    filename = (
        f"03_Calibration_"
        f"{int(t)}d"
    )

    fig.savefig(
        HOLDOUT_FIGURES_DIR /
        f"{filename}.png",
        dpi=300,
        bbox_inches="tight"
    )

    fig.savefig(
        HOLDOUT_FIGURES_DIR /
        f"{filename}.svg",
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

In [ ]:
# ============================================================
# CELL 48
# CALIBRATION SLOPE
# ============================================================

primary_lp = (
    prediction_table[
        PRIMARY_MODEL_ID
    ].values
)

cal_slope_df = pd.DataFrame({
    "OS_time":
        holdout_df[
            TIME_COL
        ].astype(float).values,

    "OS":
        holdout_df[
            EVENT_COL
        ].astype(int).values,

    "Linear_Predictor":
        primary_lp
})

cal_slope_model = CoxPHFitter()

cal_slope_model.fit(
    cal_slope_df,
    duration_col="OS_time",
    event_col="OS"
)

cal_slope_summary = pd.DataFrame([
    {
        "Calibration_Slope":
            float(
                cal_slope_model.params_[
                    "Linear_Predictor"
                ]
            ),

        "CI_Lower":
            float(
                cal_slope_model.confidence_intervals_
                .loc[
                    "Linear_Predictor"
                ]
                .iloc[0]
            ),

        "CI_Upper":
            float(
                cal_slope_model.confidence_intervals_
                .loc[
                    "Linear_Predictor"
                ]
                .iloc[1]
            ),

        "P_Value":
            float(
                cal_slope_model.summary.loc[
                    "Linear_Predictor",
                    "p"
                ]
            )
    }
])

cal_slope_summary.to_csv(
    HOLDOUT_RESULTS_DIR /
    "11_Primary_Holdout_Calibration_Slope.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    cal_slope_summary
)

In [ ]:
# ============================================================
# CELL 49
# IPCW SURVIVAL DCA
# ============================================================

def build_censoring_km(
    df
):
    km_censor = KaplanMeierFitter()

    censor_event = (
        1
        - df[
            EVENT_COL
        ].astype(int).values
    )

    km_censor.fit(
        durations=df[
            TIME_COL
        ].astype(float).values,

        event_observed=censor_event
    )

    return km_censor


def safe_G(
    km,
    t
):
    g = float(
        km.predict(
            max(
                float(t),
                0.0
            )
        )
    )

    return max(
        g,
        1e-6
    )


def ipcw_dca(
    df,
    predicted_event_probability,
    horizon,
    thresholds
):
    time = (
        df[
            TIME_COL
        ].astype(float).values
    )

    event = (
        df[
            EVENT_COL
        ].astype(int).values
    )

    pred = np.asarray(
        predicted_event_probability
    )

    n = len(df)

    censor_km = (
        build_censoring_km(
            df
        )
    )

    is_case = (
        (event == 1)
        &
        (time <= horizon)
    )

    is_control = (
        time > horizon
    )

    weights_case = np.zeros(
        n
    )

    case_indices = np.where(
        is_case
    )[0]

    for i in case_indices:

        weights_case[i] = (
            1.0
            / safe_G(
                censor_km,
                time[i] - 1e-8
            )
        )

    G_horizon = safe_G(
        censor_km,
        horizon
    )

    weights_control = (
        is_control.astype(float)
        / G_horizon
    )

    weighted_cases = (
        weights_case.sum()
        / n
    )

    weighted_controls = (
        weights_control.sum()
        / n
    )

    records = []

    for threshold in thresholds:

        positive = (
            pred >= threshold
        )

        tp = (
            weights_case[
                positive
            ].sum()
            / n
        )

        fp = (
            weights_control[
                positive
            ].sum()
            / n
        )

        odds = (
            threshold
            / (1 - threshold)
        )

        nb_model = (
            tp
            - fp * odds
        )

        nb_all = (
            weighted_cases
            - weighted_controls
            * odds
        )

        records.append({
            "Threshold":
                threshold,

            "Net_Benefit_Model":
                nb_model,

            "Net_Benefit_TreatAll":
                nb_all,

            "Net_Benefit_TreatNone":
                0.0
        })

    return pd.DataFrame(
        records
    )

In [ ]:
# ============================================================
# CELL 50
# PRIMARY HOLDOUT DCA
# ============================================================

DCA_THRESHOLDS = np.arange(
    0.05,
    0.81,
    0.01
)

dca_all = []


for j, t in enumerate(
    VALID_TIME_POINTS
):

    t = float(t)

    dca_df = ipcw_dca(
        holdout_df,
        primary_event_matrix[
            :,
            j
        ],
        horizon=t,
        thresholds=DCA_THRESHOLDS
    )

    dca_df[
        "Time_Days"
    ] = t

    dca_all.append(
        dca_df
    )


dca_all_df = pd.concat(
    dca_all,
    ignore_index=True
)

dca_all_df.to_csv(
    HOLDOUT_RESULTS_DIR /
    "12_Primary_Holdout_DCA.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    dca_all_df.head()
)

In [ ]:
# ============================================================
# CELL 51
# DCA FIGURES
# ============================================================

for t in VALID_TIME_POINTS:

    t = float(t)

    tmp = (
        dca_all_df[
            dca_all_df[
                "Time_Days"
            ] == t
        ]
    )

    fig, ax = plt.subplots(
        figsize=(7.5, 5.8)
    )

    ax.plot(
        tmp[
            "Threshold"
        ],
        tmp[
            "Net_Benefit_Model"
        ],
        linewidth=2,
        label="Primary model"
    )

    ax.plot(
        tmp[
            "Threshold"
        ],
        tmp[
            "Net_Benefit_TreatAll"
        ],
        linestyle="--",
        label="Treat all"
    )

    ax.plot(
        tmp[
            "Threshold"
        ],
        tmp[
            "Net_Benefit_TreatNone"
        ],
        linestyle=":",
        label="Treat none"
    )

    ax.set_xlabel(
        "Threshold probability"
    )

    ax.set_ylabel(
        "Net benefit"
    )

    ax.set_ylim(
        -0.5,
        0.5
    )

    ax.set_title(
        f"Holdout DCA - "
        f"{t / 365:.0f} Year"
    )

    ax.legend()

    fig.tight_layout()

    filename = (
        f"04_DCA_"
        f"{int(t)}d"
    )

    fig.savefig(
        HOLDOUT_FIGURES_DIR /
        f"{filename}.png",
        dpi=300,
        bbox_inches="tight"
    )

    fig.savefig(
        HOLDOUT_FIGURES_DIR /
        f"{filename}.svg",
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

In [ ]:
# ============================================================
# CELL 52
# APPLY LOCKED DEVELOPMENT RISK CUTOFF
# ============================================================

PRIMARY_CUTOFF = float(
    DEVELOPMENT_RISK_CUTOFF
)

prediction_table[
    "Primary_Risk_Group"
] = np.where(
    prediction_table[
        PRIMARY_MODEL_ID
    ]
    >= PRIMARY_CUTOFF,
    "High",
    "Low"
)

risk_group_counts = (
    prediction_table[
        "Primary_Risk_Group"
    ]
    .value_counts()
    .rename_axis(
        "Risk_Group"
    )
    .reset_index(
        name="N"
    )
)

risk_group_counts.to_csv(
    HOLDOUT_RESULTS_DIR /
    "13_Primary_Holdout_RiskGroup_Counts.csv",
    index=False,
    encoding="utf-8-sig"
)

prediction_table.to_csv(
    HOLDOUT_RESULTS_DIR /
    "14_Holdout_FINAL_Prediction_Table.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Frozen cutoff:",
    PRIMARY_CUTOFF
)

display(
    risk_group_counts
)

In [ ]:
# ============================================================
# CELL 53
# PRIMARY HOLDOUT KM / LOG-RANK / HR
# ============================================================

from lifelines.statistics import (
    logrank_test
)

km_data = pd.DataFrame({
    "OS_time":
        holdout_df[
            TIME_COL
        ].astype(float).values,

    "OS":
        holdout_df[
            EVENT_COL
        ].astype(int).values,

    "Risk_Group":
        prediction_table[
            "Primary_Risk_Group"
        ].values
})

high = (
    km_data[
        km_data[
            "Risk_Group"
        ] == "High"
    ]
)

low = (
    km_data[
        km_data[
            "Risk_Group"
        ] == "Low"
    ]
)


logrank_result = logrank_test(
    high[
        "OS_time"
    ],
    low[
        "OS_time"
    ],
    event_observed_A=
        high["OS"],
    event_observed_B=
        low["OS"]
)


risk_cox_df = (
    km_data.copy()
)

risk_cox_df[
    "HighRisk"
] = (
    risk_cox_df[
        "Risk_Group"
    ] == "High"
).astype(int)

risk_cox_fit = CoxPHFitter()

risk_cox_fit.fit(
    risk_cox_df[
        [
            "OS_time",
            "OS",
            "HighRisk"
        ]
    ],
    duration_col="OS_time",
    event_col="OS"
)

risk_cox_row = (
    risk_cox_fit
    .summary
    .loc[
        "HighRisk"
    ]
)

km_stats = pd.DataFrame([
    {
        "Primary_Model":
            PRIMARY_MODEL_ID,

        "Cutoff":
            PRIMARY_CUTOFF,

        "Cutoff_Source":
            "Development median",

        "High_N":
            len(high),

        "Low_N":
            len(low),

        "High_Events":
            int(
                high["OS"].sum()
            ),

        "Low_Events":
            int(
                low["OS"].sum()
            ),

        "LogRank_P":
            float(
                logrank_result.p_value
            ),

        "High_vs_Low_HR":
            float(
                np.exp(
                    risk_cox_row[
                        "coef"
                    ]
                )
            ),

        "HR_CI_Lower":
            float(
                np.exp(
                    risk_cox_row[
                        "coef lower 95%"
                    ]
                )
            ),

        "HR_CI_Upper":
            float(
                np.exp(
                    risk_cox_row[
                        "coef upper 95%"
                    ]
                )
            ),

        "Cox_P":
            float(
                risk_cox_row[
                    "p"
                ]
            )
    }
])

km_stats.to_csv(
    HOLDOUT_RESULTS_DIR /
    "15_Primary_Holdout_KM_Statistics.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    km_stats
)

In [ ]:
# ============================================================
# CELL 54
# PRIMARY HOLDOUT KM FIGURE + RAW TABLE
# ============================================================

fig, ax = plt.subplots(
    figsize=(7.5, 6)
)

km_curve_records = []


for group_name in [
    "Low",
    "High"
]:

    g = (
        km_data[
            km_data[
                "Risk_Group"
            ] == group_name
        ]
    )

    kmf = KaplanMeierFitter()

    kmf.fit(
        durations=g[
            "OS_time"
        ],
        event_observed=g[
            "OS"
        ],
        label=group_name
    )

    kmf.plot_survival_function(
        ax=ax,
        ci_show=True
    )

    sf = (
        kmf
        .survival_function_
        .reset_index()
    )

    sf.columns = [
        "Time",
        "Survival"
    ]

    sf[
        "Risk_Group"
    ] = group_name

    km_curve_records.append(
        sf
    )


km_curve_table = pd.concat(
    km_curve_records,
    ignore_index=True
)

km_curve_table.to_csv(
    HOLDOUT_RESULTS_DIR /
    "16_Primary_Holdout_KM_Curve_Data.csv",
    index=False,
    encoding="utf-8-sig"
)

ax.set_xlabel(
    "Time (days)"
)

ax.set_ylabel(
    "Overall survival probability"
)

ax.set_title(
    "Holdout OS by Frozen Development Risk Group\n"
    f"Log-rank P = "
    f"{logrank_result.p_value:.4g}"
)

fig.tight_layout()

fig.savefig(
    HOLDOUT_FIGURES_DIR /
    "05_Primary_Holdout_KM.png",
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    HOLDOUT_FIGURES_DIR /
    "05_Primary_Holdout_KM.svg",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# CELL 55
# HOLDOUT RISK SCORE DISTRIBUTION
# ============================================================

risk_plot = (
    prediction_table[
        [
            "Sample_ID",
            "OS_time",
            "OS",
            PRIMARY_MODEL_ID,
            "Primary_Risk_Group"
        ]
    ]
    .sort_values(
        PRIMARY_MODEL_ID
    )
    .reset_index(drop=True)
)

risk_plot[
    "Rank"
] = np.arange(
    1,
    len(risk_plot) + 1
)

risk_plot.to_csv(
    HOLDOUT_RESULTS_DIR /
    "17_Primary_Holdout_RiskScore_Ranked.csv",
    index=False,
    encoding="utf-8-sig"
)

fig, ax = plt.subplots(
    figsize=(9, 5)
)

ax.scatter(
    risk_plot[
        "Rank"
    ],
    risk_plot[
        PRIMARY_MODEL_ID
    ],
    s=28
)

ax.axhline(
    PRIMARY_CUTOFF,
    linestyle="--",
    label="Development cutoff"
)

ax.set_xlabel(
    "Patients ranked by risk score"
)

ax.set_ylabel(
    "Frozen model linear predictor"
)

ax.set_title(
    "Holdout Risk Score Distribution"
)

ax.legend()

fig.tight_layout()

fig.savefig(
    HOLDOUT_FIGURES_DIR /
    "06_Primary_Holdout_RiskScore_Distribution.png",
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    HOLDOUT_FIGURES_DIR /
    "06_Primary_Holdout_RiskScore_Distribution.svg",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# CELL 56
# FINAL HOLDOUT MODEL SUMMARY
# ============================================================

final_holdout_summary = (
    holdout_cindex
    .merge(
        FROZEN_MODEL_REGISTRY[
            [
                "Model_ID",
                "Algorithm",
                "N_Features",
                "Features"
            ]
        ],
        on="Model_ID",
        how="left"
    )
)

auc_wide = (
    holdout_auc
    .pivot(
        index="Model_ID",
        columns="Time_Days",
        values="AUC"
    )
    .reset_index()
)

auc_wide.columns = [
    (
        f"AUC_{int(c)}d"
        if isinstance(
            c,
            (int, float, np.integer, np.floating)
        )
        else c
    )
    for c in auc_wide.columns
]

final_holdout_summary = (
    final_holdout_summary
    .merge(
        auc_wide,
        on="Model_ID",
        how="left"
    )
)

final_holdout_summary.to_csv(
    HOLDOUT_RESULTS_DIR /
    "18_FINAL_Holdout_Model_Performance.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    final_holdout_summary.sort_values(
        "Point_C_Index",
        ascending=False
    )
)

In [ ]:
# ============================================================
# CELL 57
# DEVELOPMENT -> HOLDOUT GENERALIZATION
# ============================================================

primary_holdout_c = float(
    holdout_cindex.loc[
        holdout_cindex[
            "Model_ID"
        ] == PRIMARY_MODEL_ID,
        "Point_C_Index"
    ].iloc[0]
)

primary_holdout_low = float(
    holdout_cindex.loc[
        holdout_cindex[
            "Model_ID"
        ] == PRIMARY_MODEL_ID,
        "CI_2.5"
    ].iloc[0]
)

primary_holdout_high = float(
    holdout_cindex.loc[
        holdout_cindex[
            "Model_ID"
        ] == PRIMARY_MODEL_ID,
        "CI_97.5"
    ].iloc[0]
)

primary_generalization = pd.DataFrame([
    {
        "Metric":
            "Repeated CV C-index",

        "Development":
            float(
                unpen_cv_row[
                    "Mean_C_Index"
                ]
            ),

        "Holdout":
            np.nan
    },
    {
        "Metric":
            "Bootstrap optimism-corrected C-index",

        "Development":
            OPTIMISM_CORRECTED_CINDEX,

        "Holdout":
            np.nan
    },
    {
        "Metric":
            "Independent holdout C-index",

        "Development":
            np.nan,

        "Holdout":
            primary_holdout_c
    }
])

primary_generalization.to_csv(
    HOLDOUT_RESULTS_DIR /
    "19_Primary_Development_vs_Holdout.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Development repeated CV:",
    round(
        float(
            unpen_cv_row[
                "Mean_C_Index"
            ]
        ),
        4
    )
)

print(
    "Development optimism-corrected:",
    round(
        OPTIMISM_CORRECTED_CINDEX,
        4
    )
)

print(
    "Holdout:",
    round(
        primary_holdout_c,
        4
    ),
    f"95% CI "
    f"{primary_holdout_low:.4f}"
    f"–{primary_holdout_high:.4f}"
)

In [ ]:
# ============================================================
# CELL 58
# HOLDOUT VALIDATION AUDIT
# ============================================================

holdout_audit = []

holdout_audit.append(
    "TCGA-LIHC PHASE B HOLDOUT VALIDATION AUDIT"
)

holdout_audit.append(
    "=" * 80
)

holdout_audit.append(
    f"Development N: {len(development_df)}"
)

holdout_audit.append(
    f"Development events: "
    f"{int(development_df[EVENT_COL].sum())}"
)

holdout_audit.append(
    f"Holdout N: {len(holdout_df)}"
)

holdout_audit.append(
    f"Holdout events: "
    f"{int(holdout_df[EVENT_COL].sum())}"
)

holdout_audit.append(
    ""
)

holdout_audit.append(
    f"Holdout unlock time: "
    f"{HOLDOUT_UNLOCK_TIME}"
)

holdout_audit.append(
    ""
)

holdout_audit.append(
    "PRIMARY MODEL:"
)

holdout_audit.append(
    PRIMARY_MODEL_ID
)

holdout_audit.append(
    ""
)

holdout_audit.append(
    "Primary model feature reselection after unlock: NO"
)

holdout_audit.append(
    "Primary model coefficient refitting on Holdout: NO"
)

holdout_audit.append(
    "Hyperparameter retuning on Holdout: NO"
)

holdout_audit.append(
    "Risk cutoff optimization on Holdout: NO"
)

holdout_audit.append(
    "Preprocessing re-estimation on Holdout: NO"
)

holdout_audit.append(
    ""
)

holdout_audit.append(
    f"Frozen Development risk cutoff: "
    f"{PRIMARY_CUTOFF}"
)

holdout_audit.append(
    ""
)

holdout_audit.append(
    f"Primary Holdout C-index: "
    f"{primary_holdout_c:.4f}"
)

holdout_audit.append(
    f"Primary Holdout C-index 95% CI: "
    f"{primary_holdout_low:.4f}"
    f"–{primary_holdout_high:.4f}"
)

holdout_audit.append(
    ""
)

holdout_audit.append(
    f"Development repeated CV C-index: "
    f"{float(unpen_cv_row['Mean_C_Index']):.4f}"
)

holdout_audit.append(
    f"Development optimism-corrected C-index: "
    f"{OPTIMISM_CORRECTED_CINDEX:.4f}"
)

holdout_audit.append(
    ""
)

holdout_audit.append(
    "PHASE B HOLDOUT VALIDATION COMPLETE"
)

holdout_audit.append(
    "MODEL STATUS: FROZEN - AWAITING INTERPRETATION"
)

holdout_audit_text = "\n".join(
    holdout_audit
)

with open(
    AUDIT_DIR /
    "PHASE_B_HOLDOUT_VALIDATION_AUDIT.md",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        holdout_audit_text
    )

print(
    holdout_audit_text
)

In [ ]:
# ============================================================
# CELL 59
# OUTPUT MANIFEST
# ============================================================

output_records = []

for folder_name, folder in [
    (
        "Results",
        HOLDOUT_RESULTS_DIR
    ),
    (
        "Figures",
        HOLDOUT_FIGURES_DIR
    ),
    (
        "FrozenModels",
        FROZEN_MODELS_DIR
    )
]:

    for path in sorted(
        folder.rglob("*")
    ):

        if path.is_file():

            output_records.append({
                "Category":
                    folder_name,

                "File":
                    path.name,

                "Full_Path":
                    str(path),

                "Size_KB":
                    round(
                        path.stat().st_size
                        / 1024,
                        2
                    )
            })


output_manifest = pd.DataFrame(
    output_records
)

output_manifest.to_csv(
    AUDIT_DIR /
    "PHASE_B_OUTPUT_MANIFEST.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    output_manifest
)